# Story Assistant — Pipeline Evaluation Notebook

## What this notebook is

This is the testing and tuning ground for the Story Assistant pipeline.
Nothing here touches the backend code. All changes to prompts, logic, and 
extraction behavior happen here first. Only when we are satisfied do we 
copy the final prompts into the backend files.

## What the pipeline does (plain english)

1. User writes a scene description
2. Backend generates scene text using Groq (Llama 3.3 70B) — fast, free
3. User approves the scene
4. On approval, Gemini 2.5 Flash reads the scene and extracts structured facts:
   - New characters and their attributes
   - Updates to existing characters
   - World facts (setting, rules, atmosphere)
   - Plot threads (opened or closed)
   - Established facts (concrete things that are now true in this world)
   - Ambiguities (things mentioned but never explained — surfaced to user)
5. These facts are stored in Supabase as individual rows (one row per atomic fact)
   each tagged with the scene they came from (source_scene_id)
6. Every future generation call receives the full bible — so the LLM always
   knows the world it is writing in

## Why we test here and not in the backend

- Prompt changes here are instant — no Docker rebuild needed
- We can run 20 scenes in a loop and inspect every output
- We can read Supabase into pandas and analyze patterns across runs
- We can compare prompt versions side by side
- Once prompts are finalized here, we paste them into the backend files once

## What we are stress testing

1. **Extraction recall** — does Gemini catch everything important in a scene?
2. **Redundancy** — does it create duplicate or near-duplicate facts across scenes?
3. **Hallucination** — does it invent facts not present in the scene?
4. **Accumulation** — does the bible stay coherent and non-contradictory over 10+ scenes?
5. **Bible usage** — does Groq actually use the bible when generating new scenes?
6. **JSON reliability** — does Gemini always return valid parseable JSON?
7. **Implicit facts** — does it catch facts that are shown but not stated directly?
8. **Tone consistency** — does generated prose stay consistent in voice across scenes?
9. **Cross-scene Continuity** — plant in scene 1, verify survives to scene 6

## How to run a test iteration

1. Edit the prompt variables in the **Prompts cell**
2. Re-run the Prompts cell and the Functions cell
3. Run `clear_project_data(PROJECT_ID)` to wipe previous test data
4. Run scenes through `run_pipeline()` one by one or in a batch loop
5. Call `get_bible_facts(PROJECT_ID)` to read results into a DataFrame
6. Use the analysis cells to measure what worked and what did not
7. Adjust prompts and repeat

## Files this notebook talks to

- `../backend/.env` — reads API keys (Groq, Gemini, Supabase)
- Supabase tables: `scenes`, `bible_facts`, `bible_questions`
- Does NOT import anything from the backend Python code — fully standalone

## When we are done here

Copy the final versions of these into the backend:
- `EXTRACTION_SYSTEM_PROMPT` + `EXTRACTION_PROMPT_TEMPLATE` → `backend/app/services/llm.py`
- `GENERATION_SYSTEM_PROMPT` + `GENERATION_PROMPT_TEMPLATE` → `backend/app/services/llm.py`

# Testing Plan

## Phase 1 — Baseline (run first, touch nothing)
Run 3 scenes from the same story through the pipeline as-is.
Goal: understand current behavior before tuning anything.
Measure: row count per scene, any obvious errors, JSON parse failures.

## Phase 2 — Extraction stress tests
Run 6 targeted scenes designed to stress specific failure modes.
Each scene is crafted to test one thing.

## Phase 3 — Accumulation test
Run 8 scenes in sequence from the same story.
Goal: does the bible stay coherent? Does redundancy grow?
Measure: duplicate facts, contradictions, row count growth curve.

## Phase 4 — Generation quality test
Generate scenes using an accumulated bible.
Goal: does Groq actually use the bible? Does tone hold?
Measure: manual read + checklist.

## Phase 5 — Prompt tuning
Based on Phase 2-4 findings, edit prompts and re-run the specific
failing tests. Repeat until satisfied. Then re-run full Phase 3 to
confirm the fix did not break anything else.

# Pipeline functions

In [9]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from groq import Groq
import google.generativeai as genai
from supabase import create_client

# Load your existing .env from backend folder
load_dotenv("../backend/.env")

# Clients
groq_client = Groq(api_key=os.getenv("GROQ_KEY"))
genai.configure(api_key=os.getenv("GEMINI_KEY"))

supabase = create_client(
    os.getenv("SUPABASE_URL"),
    os.getenv("SUPABASE_KEY")
)

print("All clients initialized")

All clients initialized


In [2]:
def get_bible_facts(project_id: str) -> pd.DataFrame:
    result = supabase.table("bible_facts")\
        .select("*")\
        .eq("project_id", project_id)\
        .execute()
    return pd.DataFrame(result.data)

def get_scenes(project_id: str) -> pd.DataFrame:
    result = supabase.table("scenes")\
        .select("*")\
        .eq("project_id", project_id)\
        .execute()
    return pd.DataFrame(result.data)

def clear_project_data(project_id: str):
    """Wipe all facts and scenes for a project — use between test runs."""
    supabase.table("bible_facts").delete().eq("project_id", project_id).execute()
    supabase.table("bible_questions").delete().eq("project_id", project_id).execute()
    supabase.table("scenes").delete().eq("project_id", project_id).execute()
    print(f"Cleared all data for project {project_id}")

print("Supabase helpers ready")

Supabase helpers ready


In [25]:
# ---- TUNE THESE WITHOUT TOUCHING BACKEND FILES ----

EXTRACTION_SYSTEM_PROMPT = """You are a precise story analyst. 
Extract structured facts from fiction. 
Return only valid JSON with no markdown fences or extra text.
Be selective — only extract facts that are clearly established in the scene, 
not things that are implied or assumed. Avoid redundant facts. Established facts are PERMANENT truths about the world, characters, or story — 
things that will still be true in future scenes. They are NOT scene actions or 
moment-to-moment events. "Elena cannot drive" is an established fact. 
"Elena walked to the bar" is not. Exception: always extract specific physical descriptions of characters 
(height, build, distinguishing features, clothing details that identify them) 
even if they seem like scene details — these are needed for visual consistency 
in future scenes."""

EXTRACTION_PROMPT_TEMPLATE = """EXISTING STORY BIBLE:
{existing_bible}

NEW SCENE TO ANALYZE:
{scene_content}

Extract new information from this scene. Return ONLY valid JSON.
Avoid duplicating facts already in the story bible.

{{
  "new_characters": [
    {{
      "name": "character name",
      "physical": "physical description if mentioned",
      "personality": "personality traits if shown",
      "role": "their role in the story",
      "relationships": "relationships to other characters"
    }}
  ],
  "character_updates": [
    {{
      "name": "existing character name",
      "new_info": "what new thing we learned about them"
    }}
  ],
  "world_facts": [
    "ONE fact per item, no redundancy"
  ],
  "plot_threads": [
    {{
      "thread": "description of plot thread",
      "status": "opened or closed",
      "clues": ["any clues dropped"]
    }}
  ],
  "established_facts": [
    "permanent facts about the world or characters that future scenes must respect — NOT moment-to-moment actions or events"
  ],
  "ambiguities": [
    "things mentioned but not explained"
  ]
}}"""

GENERATION_SYSTEM_PROMPT = """You are a literary writing assistant. 
Write in the established tone and style of this story. 
Never introduce facts that contradict the story bible.
Always reference specific details from the story bible to maintain consistency."""

GENERATION_PROMPT_TEMPLATE = """STORY BIBLE:
{bible_text}
{last_scene_section}

USER'S SCENE DESCRIPTION:
{description}

Write this scene in 2-3 literary paragraphs. Match the established tone precisely. 
Do not introduce new characters unless specified in the description.
Do not contradict any facts in the story bible."""

print("Prompts loaded — edit these cells freely")

Prompts loaded — edit these cells freely
The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [4]:
def format_bible_for_prompt(bible_facts: list) -> str:
    if not bible_facts:
        return "No story bible established yet."
    
    sections = {"character": [], "world": [], "plot_thread": [], "established_fact": []}
    for fact in bible_facts:
        category = fact.get("category", "established_fact")
        content = fact.get("content", {})
        if category in sections:
            sections[category].append(content)
    
    output = []
    if sections["character"]:
        output.append("CHARACTERS:\n" + json.dumps(sections["character"], indent=2))
    if sections["world"]:
        output.append("WORLD RULES:\n" + json.dumps(sections["world"], indent=2))
    if sections["plot_thread"]:
        output.append("PLOT THREADS:\n" + json.dumps(sections["plot_thread"], indent=2))
    if sections["established_fact"]:
        output.append("ESTABLISHED FACTS:\n" + json.dumps(sections["established_fact"], indent=2))
    
    return "\n\n".join(output)


def extract_facts(scene_content: str, existing_facts: list) -> dict:
    existing_text = format_bible_for_prompt(existing_facts)
    prompt = EXTRACTION_PROMPT_TEMPLATE.format(
        existing_bible=existing_text,
        scene_content=scene_content
    )
    model = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=EXTRACTION_SYSTEM_PROMPT
    )
    response = model.generate_content(prompt)
    raw = response.text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw.strip())


def generate_scene(description: str, bible_facts: list, last_scene: str = "") -> str:
    bible_text = format_bible_for_prompt(bible_facts)
    last_scene_section = f"\n\nLAST APPROVED SCENE:\n{last_scene}" if last_scene else ""
    prompt = GENERATION_PROMPT_TEMPLATE.format(
        bible_text=bible_text,
        last_scene_section=last_scene_section,
        description=description
    )
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": GENERATION_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        max_tokens=1024,
        temperature=0.7
    )
    return response.choices[0].message.content


print("Core functions ready")

Core functions ready


In [5]:
def run_pipeline(project_id: str, scene_number: int, scene_content: str, 
                 save_to_supabase: bool = True) -> dict:
    """
    Simulates the full approve flow:
    1. Fetch existing bible from Supabase
    2. Extract facts from new scene
    3. Optionally write to Supabase
    Returns extracted facts for inspection.
    """
    # Fetch existing bible
    existing = supabase.table("bible_facts")\
        .select("*")\
        .eq("project_id", project_id)\
        .eq("status", "confirmed")\
        .execute()
    existing_facts = existing.data or []
    
    print(f"Existing bible: {len(existing_facts)} facts")
    
    # Extract
    extracted = extract_facts(scene_content, existing_facts)
    
    print(f"Extracted: {json.dumps(extracted, indent=2)}")
    
    if save_to_supabase:
        # Save scene
        scene_result = supabase.table("scenes").insert({
            "project_id": project_id,
            "scene_number": scene_number,
            "version": 1,
            "content": scene_content,
            "status": "active"
        }).execute()
        scene_id = scene_result.data[0]["id"]
        
        # Save facts
        rows = []
        for char in extracted.get("new_characters", []):
            rows.append({"project_id": project_id, "source_scene_id": scene_id,
                        "category": "character", "content": char, "status": "confirmed"})
        for update in extracted.get("character_updates", []):
            rows.append({"project_id": project_id, "source_scene_id": scene_id,
                        "category": "character", "content": update, "status": "confirmed"})
        for fact in extracted.get("world_facts", []):
            rows.append({"project_id": project_id, "source_scene_id": scene_id,
                        "category": "world", "content": {"fact": fact}, "status": "confirmed"})
        for thread in extracted.get("plot_threads", []):
            rows.append({"project_id": project_id, "source_scene_id": scene_id,
                        "category": "plot_thread", "content": thread, "status": "confirmed"})
        for fact in extracted.get("established_facts", []):
            rows.append({"project_id": project_id, "source_scene_id": scene_id,
                        "category": "established_fact", "content": {"fact": fact}, "status": "confirmed"})
        
        if rows:
            supabase.table("bible_facts").insert(rows).execute()
        
        if extracted.get("ambiguities"):
            supabase.table("bible_questions").insert({
                "project_id": project_id,
                "source_scene_id": scene_id,
                "questions": extracted["ambiguities"],
                "status": "pending"
            }).execute()
        
        print(f"Saved {len(rows)} facts to Supabase")
    
    return extracted


print("Pipeline ready")

Pipeline ready


In [6]:
# Use your existing test project or create a new one
PROJECT_ID = "73a1bada-b2bf-423b-8c6e-f107076da2ed"

# Clear it before starting fresh tests
clear_project_data(PROJECT_ID)

Cleared all data for project 73a1bada-b2bf-423b-8c6e-f107076da2ed


# Testing

## How this section works
Each test has:
- A clearly labelled scene or set of scenes
- Exactly what facts are planted (so we know what to look for)
- The code to run it
- The code to evaluate it
- A pass/fail criteria

## Test index
1. Extraction Recall — planted facts, scored
2. Redundancy — word overlap analysis across rows  
3. Hallucination — subtext scene, label each output fact
4. Accumulation — row growth curve over 8 sequential scenes
5. Bible Usage — with vs without bible comparison
6. JSON Reliability — 20 rapid extractions, count failures
7. Implicit Facts — behavior-shown facts, labeled explicit vs implicit
8. Tone Consistency — sentence length + word class analysis
9. Cross-scene Continuity — plant in scene 1, verify survives to scene 6

## Ground rules
- Clear Supabase before each full test run
- Change ONE prompt variable at a time between iterations
- Record what you changed and what improved in the notes cell at the bottom
- Do not touch backend files until all 9 pass satisfactorily

In [26]:
# ═══════════════════════════════════════════════════════════════
# ALL TEST SCENES — single source of truth
# Each scene has: label, content, planted_facts (what SHOULD appear),
# and test_targets (which of the 9 tests it serves)
# ═══════════════════════════════════════════════════════════════

TEST_SCENES = [

    # ── SCENE 1 ─────────────────────────────────────────────────
    # Tests: recall, cross-scene continuity (plant), accumulation
    # Planted facts: 6 explicit, 1 continuity anchor
    {
        "scene_number": 1,
        "label": "opening_intro",
        "test_targets": [1, 4, 9],
        "planted_facts": [
            "Elena has red hair",
            "Elena has a scar on her left hand",        # continuity anchor for test 9
            "Elena cannot drive",
            "Elena lives in Zizkov",
            "The city is Prague",
            "The year is 2041",
        ],
        "content": """Elena arrived at the dojo just after midnight. She had walked the four 
kilometers from her apartment in Zizkov — she could not drive, never had learned. The 
building was still smoldering, ash drifting through the cold Prague air like grey snow. 
She crouched near the entrance, her red hair damp from the rain, and found it near the 
door frame: a matchbook, embossed with a symbol she recognised. As she picked it up her 
left hand caught the light — the scar running across the palm, pale and old, from 
something she did not talk about. The year was 2041 and Prague smelled like smoke 
more often than it used to."""
    },

    # ── SCENE 2 ─────────────────────────────────────────────────
    # Tests: recall, accumulation
    # Planted facts: 5 explicit
    {
        "scene_number": 2,
        "label": "bar_scene",
        "test_targets": [1, 4],
        "planted_facts": [
            "Bar is called Cerny Kohout",
            "Tomas is a bartender",
            "Tomas is the only person Elena trusts in Prague",
            "Kova Corp controls the water supply",
            "Kova Corp bought the dojo building last month",
        ],
        "content": """The bar was called Cerny Kohout — the Black Rooster — and it had not 
changed since 1987. Tomas slid a glass across the counter without being asked. He was 
the only person in Prague Elena trusted, which was not saying much. 
'Kova Corp bought the building last month,' he said. 'Three days after the purchase, 
the fire inspector stopped coming around.' 
Elena turned the matchbook over in her fingers. Kova Corp controlled the water supply 
and half the city council. Now apparently they were buying dojos in Zizkov."""
    },

    # ── SCENE 3 ─────────────────────────────────────────────────
    # Tests: recall, accumulation, implicit facts
    # Planted facts: 3 explicit, 3 implicit (shown through behavior)
    {
        "scene_number": 3,
        "label": "metro_implicit",
        "test_targets": [1, 4, 7],
        "planted_facts": [
            "Elena pays with coins late at night, never card",   # implicit — shown behavior
            "Elena always sits facing backward on the metro",    # implicit — stated as habit
            "Elena ignores calls from unknown numbers",          # implicit — shown behavior
            "Elena takes the metro as her main transport",       # explicit
            "The metro runs past midnight in Prague 2041",       # explicit
            "Elena has old cautious habits",                     # implicit — inference from behavior
        ],
        "content": """The metro was still running at this hour, which surprised her. Elena 
bought a ticket with coins — she never used her card this late, old habit from years she 
preferred not to think about. She sat in the last car, facing backward the way she always 
did, watching the tunnel walls blur past. The city moved differently at night, quieter, 
more honest about what it was. Her phone buzzed. Unknown number. She watched it ring 
until it stopped, then put the phone face down on her knee."""
    },

    # ── SCENE 4 ─────────────────────────────────────────────────
    # Tests: recall, accumulation, many characters at once
    # Planted facts: 3 new characters with distinct attributes
    {
        "scene_number": 4,
        "label": "three_characters",
        "test_targets": [1, 4],
        "planted_facts": [
            "Vera works in Kova Corp legal department",
            "Vera is terrified",
            "The Mechanic is heavyset and says nothing",
            "The Mechanic watches exits",
            "The boy is approximately sixteen years old",
            "The boy has a shaved head",
            "The boy sent Elena the photograph",
            "The boy wears a Kova Corp badge upside down",
        ],
        "content": """The meeting was in a basement off Narodni. Three people she had never met: 
a woman called Vera who worked in the Kova Corp legal department and was clearly terrified — 
she had not touched her coffee, her hands were flat on the table as if she was steadying 
herself against something. A heavyset man introduced only as the Mechanic, who said nothing 
and watched the door the entire time, shifting his weight slightly each time someone walked 
past in the corridor above. And a teenager, maybe sixteen, with a shaved head and a Kova 
Corp employee badge clipped upside down to his jacket — the boy who had sent her the 
photograph."""
    },

    # ── SCENE 5 ─────────────────────────────────────────────────
    # Tests: hallucination, accumulation
    # Almost no extractable facts — pure mood and interiority
    # Anything extracted beyond the 2 facts below is a hallucination candidate
    {
        "scene_number": 5,
        "label": "subtext_only",
        "test_targets": [3, 4],
        "planted_facts": [
            "Elena has not contacted her father in fifteen years",  # only real fact
            "Elena stopped smoking three years ago",               # only real fact
        ],
        "content": """She did not sleep. She sat by the window and watched the rain and thought 
about her father. Not the man in the photograph — the younger one, the one who smiled — 
but the man she remembered: quiet, careful, always leaving a room slightly before anyone 
noticed. She had spent fifteen years being angry at him. She was not sure anymore what 
she was feeling. On the sill there was an old ashtray she had never thrown away, though 
she had not smoked in three years. Outside, the rain did not stop."""
    },

    # ── SCENE 6 ─────────────────────────────────────────────────
    # Tests: cross-scene continuity, bible usage
    # References scene 1 planted facts — scar on LEFT hand specifically
    # If bible preserved it correctly, generation should say left hand
    {
        "scene_number": 6,
        "label": "continuity_test",
        "test_targets": [5, 9],
        "planted_facts": [
            "Elena bandages her left hand",   # should match scene 1 scar location
        ],
        "content": """The cut was not deep but it bled more than it should have. Elena wrapped 
her hand in a strip torn from her jacket lining, pulling the knot tight with her teeth. 
The left hand — always the left hand, the same palm, the old scar tissue not holding as 
well as new skin would have. She had reopened it on the broken window frame. She did not 
have time to do this properly."""
    },

    # ── SCENE 7 ─────────────────────────────────────────────────
    # Tests: world building density, redundancy risk, accumulation
    # Dense with world facts — tests whether extraction inflates or stays precise
    {
        "scene_number": 7,
        "label": "world_building_dense",
        "test_targets": [2, 4],
        "planted_facts": [
            "Prague floods every year between October and March",
            "The Vltava flooded three times in the last decade",
            "Lower districts were rebuilt on raised platforms after second flood",
            "Kova Corp funded the post-flood reconstruction",
            "Holesovice, Karlin and Zizkov are lower districts",
            "The city is structurally indebted to Kova Corp",
        ],
        "content": """In 2041 Prague the rain never really stopped between October and March. 
The Vltava had flooded three times in the last decade, each time worse, and the lower 
districts — Holesovice, Karlin, parts of Zizkov — had been rebuilt on raised platforms 
after the second flood. Kova Corp had funded the reconstruction. Everyone knew what that 
meant. The city owed them and the debt was structural, literally built into the 
foundations. You could see it if you knew where to look: the new concrete was a slightly 
different grey than the old stone, and everywhere the new grey touched the old stone, 
Kova Corp had a claim."""
    },

    # ── SCENE 8 ─────────────────────────────────────────────────
    # Tests: redundancy (runs after 7 rich scenes — will duplicates appear?)
    # Also tests accumulation endpoint
    # Deliberately re-mentions facts already in bible
    {
        "scene_number": 8,
        "label": "redundancy_test",
        "test_targets": [2, 4],
        "planted_facts": [
            "File room is on the third floor",           # new fact
            "A photograph shows Elenas father at the dojo",  # new fact
            "The unknown man in the photo is not yet identified",  # new plot thread
        ],
        "content": """She took the stairs two at a time, the old wood groaning under her boots. 
The file room was on the third floor of the Kova Corp building — the same Kova Corp that 
had bought the dojo in Zizkov, the same company that controlled the water supply. The 
files were gone. All of them. But on the floor, half under the radiator, was a photograph: 
a man she did not recognise standing next to her father, both of them younger, both of 
them smiling, in front of the dojo. Elena picked it up with her left hand. She walked 
back out into the Prague night."""
    },
]

print(f"Loaded {len(TEST_SCENES)} test scenes")
print(f"Total planted facts across all scenes: {sum(len(s['planted_facts']) for s in TEST_SCENES)}")

Task was destroyed but it is pending!
task: <Task pending name='Task-137' coro=<_async_in_context.<locals>.run_in_context() done, defined at /opt/anaconda3/envs/story-assistant/lib/python3.11/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-138' coro=<Kernel.shell_main() running at /opt/anaconda3/envs/story-assistant/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /opt/anaconda3/envs/story-assistant/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py:563]>
/opt/anaconda3/envs/story-assistant/lib/python3.11/tokenize.py:529: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  pseudomatch = _compile(PseudoToken).match(line, pos)
Task was destroyed but it is pending!
task: <Task pending name='Task-138' coro=<Kernel.shell_main() running at /opt/anaconda3/envs/story-assistant/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]>
Task was

Loaded 8 test scenes
Total planted facts across all scenes: 37


## Scene Testing — Iteration 1


In [11]:
# ═══════════════════════════════════════════════════════════════
# SETUP — clear DB and run all 8 scenes through full pipeline
# Gemini free tier: 5 req/min → 12s minimum between calls
# With retry logic for quota errors
# ═══════════════════════════════════════════════════════════════

import re as _re
import time

def extract_facts_with_retry(scene_content, existing_facts, max_retries=3):
    """Wraps extract_facts with retry logic for rate limit errors."""
    for attempt in range(max_retries):
        try:
            return extract_facts(scene_content, existing_facts)
        except Exception as e:
            err = str(e)
            if "429" in err or "RESOURCE_EXHAUSTED" in err or "quota" in err.lower():
                # Try to parse wait time from error message
                match = _re.search(r'retry in (\d+)', err)
                wait = int(match.group(1)) + 3 if match else 35
                print(f"  Rate limited. Waiting {wait}s before retry {attempt+1}/{max_retries}...")
                time.sleep(wait)
            else:
                raise  # non-rate-limit error, don't retry
    raise Exception("Max retries exceeded")

def run_pipeline_safe(project_id, scene_number, scene_content, save_to_supabase=True):
    """Same as run_pipeline but uses retry-aware extraction."""
    existing = supabase.table("bible_facts")\
        .select("*")\
        .eq("project_id", project_id)\
        .eq("status", "confirmed")\
        .execute()
    existing_facts = existing.data or []
    print(f"  Existing bible: {len(existing_facts)} facts")

    extracted = extract_facts_with_retry(scene_content, existing_facts)
    print(f"  Extracted: {json.dumps(extracted, indent=2)}")

    if save_to_supabase:
        scene_result = supabase.table("scenes").insert({
            "project_id": project_id,
            "scene_number": scene_number,
            "version": 1,
            "content": scene_content,
            "status": "active"
        }).execute()
        scene_id = scene_result.data[0]["id"]

        rows = []
        for char in extracted.get("new_characters", []):
            rows.append({"project_id": project_id, "source_scene_id": scene_id,
                        "category": "character", "content": char, "status": "confirmed"})
        for update in extracted.get("character_updates", []):
            rows.append({"project_id": project_id, "source_scene_id": scene_id,
                        "category": "character", "content": update, "status": "confirmed"})
        for fact in extracted.get("world_facts", []):
            rows.append({"project_id": project_id, "source_scene_id": scene_id,
                        "category": "world", "content": {"fact": fact}, "status": "confirmed"})
        for thread in extracted.get("plot_threads", []):
            rows.append({"project_id": project_id, "source_scene_id": scene_id,
                        "category": "plot_thread", "content": thread, "status": "confirmed"})
        for fact in extracted.get("established_facts", []):
            rows.append({"project_id": project_id, "source_scene_id": scene_id,
                        "category": "established_fact", "content": {"fact": fact}, "status": "confirmed"})

        if rows:
            supabase.table("bible_facts").insert(rows).execute()

        if extracted.get("ambiguities"):
            supabase.table("bible_questions").insert({
                "project_id": project_id,
                "source_scene_id": scene_id,
                "questions": extracted["ambiguities"],
                "status": "pending"
            }).execute()

        print(f"  Saved {len(rows)} facts to Supabase")

    return extracted

# ── MAIN SETUP RUN ───────────────────────────────────────────
RATE_LIMIT_DELAY = 13  # 12s minimum + 1s buffer for 5 req/min

clear_project_data(PROJECT_ID)
print("DB cleared\n")

pipeline_results = {}

for i, scene in enumerate(TEST_SCENES):
    print(f"Scene {scene['scene_number']}/{len(TEST_SCENES)}: {scene['label']}")
    extracted = run_pipeline_safe(PROJECT_ID, scene["scene_number"], scene["content"])
    pipeline_results[scene["label"]] = extracted

    if i < len(TEST_SCENES) - 1:
        print(f"  waiting {RATE_LIMIT_DELAY}s...\n")
        time.sleep(RATE_LIMIT_DELAY)

print(f"\n{'='*50}")
print(f"All {len(TEST_SCENES)} scenes processed")
print(f"pipeline_results keys: {list(pipeline_results.keys())}")
print(f"Total expected runtime: ~{len(TEST_SCENES) * RATE_LIMIT_DELAY}s")

Cleared all data for project 73a1bada-b2bf-423b-8c6e-f107076da2ed
DB cleared

Scene 1/8: opening_intro
  Existing bible: 0 facts
  Extracted: {
  "new_characters": [
    {
      "name": "Elena",
      "physical": "red hair, has a pale and old scar running across her left palm",
      "personality": "does not talk about the origin of her scar, suggesting a guarded nature",
      "role": "main character",
      "relationships": null
    }
  ],
  "character_updates": [],
  "world_facts": [
    "The year is 2041.",
    "Prague is a city where the air is cold.",
    "Prague smells like smoke more often than it used to.",
    "Zizkov is a location in Prague.",
    "There is a building referred to as a dojo."
  ],
  "plot_threads": [
    {
      "thread": "An investigation into the dojo's condition.",
      "status": "opened",
      "clues": [
        "The dojo building was still smoldering.",
        "Ash was drifting through the air near the dojo.",
        "A matchbook embossed with a reco

In [12]:
# ═══════════════════════════════════════════════════════════════
# TEST 6 — JSON RELIABILITY
# One extraction per scene (8 total) with rate limit respect
# Gemini free tier: 10 requests/min → 6s minimum between calls
# ═══════════════════════════════════════════════════════════════

import time

json_results = []
RATE_LIMIT_DELAY = 6  # seconds — keeps us under 10 req/min

print("Running 8 extractions for JSON reliability (6s delay between calls)...\n")

for i, scene in enumerate(TEST_SCENES):
    try:
        result = extract_facts(scene["content"], [])
        json_results.append({
            "scene": scene["label"], 
            "run": i+1, 
            "status": "pass", 
            "error": None
        })
        print(f"  Run {i+1:02d} [{scene['label']}]: PASS")
    except json.JSONDecodeError as e:
        json_results.append({
            "scene": scene["label"], 
            "run": i+1, 
            "status": "fail", 
            "error": f"JSON parse error: {str(e)[:80]}"
        })
        print(f"  Run {i+1:02d} [{scene['label']}]: FAIL (JSON) — {str(e)[:60]}")
    except Exception as e:
        json_results.append({
            "scene": scene["label"], 
            "run": i+1, 
            "status": "fail", 
            "error": str(e)[:80]
        })
        print(f"  Run {i+1:02d} [{scene['label']}]: FAIL (other) — {str(e)[:60]}")
    
    if i < len(TEST_SCENES) - 1:  # no delay after last one
        print(f"         waiting {RATE_LIMIT_DELAY}s...")
        time.sleep(RATE_LIMIT_DELAY)

passed = sum(1 for r in json_results if r["status"] == "pass")
json_failures = [r for r in json_results if r["status"] == "fail" 
                 and "JSON" in r.get("error", "")]
other_failures = [r for r in json_results if r["status"] == "fail" 
                  and "JSON" not in r.get("error", "")]

print(f"\n── JSON RELIABILITY RESULT ──")
print(f"Passed:            {passed}/8")
print(f"JSON parse errors: {len(json_failures)}  ← actual reliability failures")
print(f"Other errors:      {len(other_failures)}  ← rate limits, network, etc (not counted)")

if len(json_failures) == 0:
    print("✓ PASS — JSON is reliable, safe to proceed")
else:
    print("✗ FAIL — JSON parsing unreliable, fix extraction prompt")
    for f in json_failures:
        print(f"  Failed: {f['scene']} — {f['error']}")

Running 8 extractions for JSON reliability (6s delay between calls)...

  Run 01 [opening_intro]: PASS
         waiting 6s...
  Run 02 [bar_scene]: PASS
         waiting 6s...
  Run 03 [metro_implicit]: PASS
         waiting 6s...
  Run 04 [three_characters]: PASS
         waiting 6s...
  Run 05 [subtext_only]: PASS
         waiting 6s...
  Run 06 [continuity_test]: PASS
         waiting 6s...
  Run 07 [world_building_dense]: PASS
         waiting 6s...
  Run 08 [redundancy_test]: PASS

── JSON RELIABILITY RESULT ──
Passed:            8/8
JSON parse errors: 0  ← actual reliability failures
Other errors:      0  ← rate limits, network, etc (not counted)
✓ PASS — JSON is reliable, safe to proceed


In [13]:
# ═══════════════════════════════════════════════════════════════
# TEST 1 — EXTRACTION RECALL
# For each scene tagged with test_target 1:
#   list planted facts, check which appeared in extracted output
#   score = found / planted per scene, then overall
# Word overlap >= 0.5 counts as "found"
# ═══════════════════════════════════════════════════════════════

def facts_to_flat_string(extracted: dict) -> str:
    parts = []
    for key, val in extracted.items():
        if isinstance(val, list):
            for item in val:
                if isinstance(item, dict):
                    parts.append(json.dumps(item).lower())
                else:
                    parts.append(str(item).lower())
    return " ".join(parts)

def word_overlap(a: str, b: str) -> float:
    words_a = set(a.lower().split())
    words_b = set(b.lower().split())
    if not words_a:
        return 0.0
    return len(words_a & words_b) / len(words_a)

recall_results = []
recall_scenes = [s for s in TEST_SCENES if 1 in s["test_targets"]]

for scene in recall_scenes:
    extracted_flat = facts_to_flat_string(pipeline_results[scene["label"]])
    found = []
    missed = []
    for fact in scene["planted_facts"]:
        overlap = word_overlap(fact, extracted_flat)
        if overlap >= 0.5:
            found.append(fact)
        else:
            missed.append(fact)
    recall = len(found) / len(scene["planted_facts"])
    recall_results.append({
        "scene": scene["label"],
        "planted": len(scene["planted_facts"]),
        "found": len(found),
        "missed": missed,
        "recall": recall
    })

print("── TEST 1: EXTRACTION RECALL ──\n")
total_planted = sum(r["planted"] for r in recall_results)
total_found = sum(r["found"] for r in recall_results)

for r in recall_results:
    status = "✓" if r["recall"] >= 0.8 else "✗"
    print(f"{status} {r['scene']:<25} {r['found']}/{r['planted']} ({r['recall']*100:.0f}%)")
    if r["missed"]:
        for m in r["missed"]:
            print(f"    MISSED: {m}")

overall = total_found / total_planted
print(f"\nOVERALL RECALL: {total_found}/{total_planted} ({overall*100:.0f}%)")
print(f"TARGET: 80%+")
print(f"RESULT: {'✓ PASS' if overall >= 0.8 else '✗ FAIL — improve extraction prompt'}")

── TEST 1: EXTRACTION RECALL ──

✓ opening_intro             6/6 (100%)
✓ bar_scene                 5/5 (100%)
✓ metro_implicit            5/6 (83%)
    MISSED: Elena has old cautious habits
✓ three_characters          7/8 (88%)
    MISSED: The Mechanic is heavyset and says nothing

OVERALL RECALL: 23/25 (92%)
TARGET: 80%+
RESULT: ✓ PASS


In [14]:
# ═══════════════════════════════════════════════════════════════
# TEST 2 — REDUNDANCY
# Compare every pair of facts within same category
# Flag pairs with >70% word overlap as likely duplicates
# Scene 7 and 8 are the main risk — they re-mention established facts
# ═══════════════════════════════════════════════════════════════

def content_to_string(content) -> str:
    if isinstance(content, dict):
        return " ".join(str(v) for v in content.values()).lower()
    return str(content).lower()

df = get_bible_facts(PROJECT_ID)
redundant_pairs = []

for category in ["character", "world", "plot_thread", "established_fact"]:
    cat_rows = df[df["category"] == category].copy()
    cat_rows["content_str"] = cat_rows["content"].apply(content_to_string)
    rows = cat_rows.to_dict("records")
    for i in range(len(rows)):
        for j in range(i+1, len(rows)):
            overlap = word_overlap(rows[i]["content_str"], rows[j]["content_str"])
            if overlap >= 0.7:
                redundant_pairs.append({
                    "category": category,
                    "overlap": round(overlap, 2),
                    "fact_a": rows[i]["content_str"][:120],
                    "fact_b": rows[j]["content_str"][:120],
                })

print("── TEST 2: REDUNDANCY ──\n")
print(f"Total bible facts in DB: {len(df)}")
print(f"Breakdown by category:")
for cat, count in df["category"].value_counts().items():
    print(f"  {cat:<20} {count} rows")

print(f"\nSuspected redundant pairs (>70% word overlap): {len(redundant_pairs)}")

if redundant_pairs:
    print("\nRedundant pairs:")
    for p in redundant_pairs:
        print(f"\n  [{p['category']}] overlap={p['overlap']}")
        print(f"  A: {p['fact_a']}")
        print(f"  B: {p['fact_b']}")
else:
    print("No redundant pairs found")

print(f"\nTARGET: 0 pairs above 70% overlap")
print(f"RESULT: {'✓ PASS' if len(redundant_pairs) == 0 else f'✗ FAIL — {len(redundant_pairs)} redundant pairs found'}")

── TEST 2: REDUNDANCY ──

Total bible facts in DB: 101
Breakdown by category:
  established_fact     63 rows
  character            18 rows
  world                14 rows
  plot_thread          6 rows

Suspected redundant pairs (>70% word overlap): 11

Redundant pairs:

  [world] overlap=0.83
  A: zizkov is a location in prague.
  B: narodni is a location in prague.

  [world] overlap=0.83
  A: zizkov is a location in prague.
  B: holesovice is a location in prague.

  [world] overlap=0.83
  A: zizkov is a location in prague.
  B: karlin is a location in prague.

  [world] overlap=0.83
  A: narodni is a location in prague.
  B: holesovice is a location in prague.

  [world] overlap=0.83
  A: narodni is a location in prague.
  B: karlin is a location in prague.

  [world] overlap=0.83
  A: holesovice is a location in prague.
  B: karlin is a location in prague.

  [established_fact] overlap=0.83
  A: elena had never met vera before.
  B: elena had never met the mechanic before.

  [esta

In [16]:
# ═══════════════════════════════════════════════════════════════
# TEST 3 — HALLUCINATION (part 1 of 2)
# Scene 5 (subtext_only) has almost no extractable facts
# Only 2 facts were planted — everything else is a hallucination candidate
# Read the output carefully then fill in labels in the next cell
# ═══════════════════════════════════════════════════════════════

subtext_extracted = pipeline_results["subtext_only"]

print("── TEST 3: HALLUCINATION ──")
print("Scene: subtext_only")
print("\nOnly 2 real facts were planted in this scene:")
print("  1. Elena has not contacted her father in fifteen years")
print("  2. Elena stopped smoking three years ago")
print("\nFull extracted output:")
print(json.dumps(subtext_extracted, indent=2))

print("\n── ALL FACTS EXTRACTED FROM THIS SCENE ──")
print("Read each one and decide: is it grounded in the text, or invented?\n")

all_facts_from_subtext = []
for key in ["new_characters", "character_updates", "world_facts",
            "plot_threads", "established_facts"]:
    items = subtext_extracted.get(key, [])
    for item in items:
        fact_str = json.dumps(item) if isinstance(item, dict) else str(item)
        all_facts_from_subtext.append(fact_str)
        print(f"  [ ] {fact_str[:140]}")

print(f"\nTotal extracted: {len(all_facts_from_subtext)}")
print("\nNow fill in hallucination_labels in the next cell and run it")

── TEST 3: HALLUCINATION ──
Scene: subtext_only

Only 2 real facts were planted in this scene:
  1. Elena has not contacted her father in fifteen years
  2. Elena stopped smoking three years ago

Full extracted output:
{
  "new_characters": [
    {
      "name": "Father",
      "physical": "In a photograph, he appeared younger and smiled.",
      "personality": "Quiet, careful, always leaving a room slightly before anyone noticed (as remembered by Elena).",
      "role": "Elena's father",
      "relationships": "Father to Elena."
    }
  ],
  "character_updates": [
    {
      "name": "Elena",
      "new_info": "Has a long-standing emotional history with her father, including fifteen years of anger, and is currently unsure of her feelings towards him. Previously smoked but quit three years ago."
    }
  ],
  "world_facts": [],
  "plot_threads": [],
  "established_facts": [
    "Elena did not sleep.",
    "Elena sat by her window.",
    "Elena watched the rain.",
    "Elena thought abou

In [17]:
hallucination_labels = {
    '{"name": "Elena\'s Father", "physical": "appears younger and smiles in a photograph", "personality": "quiet, careful, always leaving a room slightly before anyone noticed (as remembered by Elena)", "role": "Elena\'s father", "relationships": "father of Elena"}': 1,
    '{"name": "Elena", "new_info": "Did not sleep."}': 1,
    '{"name": "Elena", "new_info": "Has a father she thinks about."}': 1,
    '{"name": "Elena", "new_info": "Spent fifteen years being angry at her father."}': 1,
    '{"name": "Elena", "new_info": "Is now unsure what she feels about her father."}': 1,
    '{"name": "Elena", "new_info": "Has not smoked in three years."}': 1,
    '{"thread": "Elena\'s complicated relationship with her father, including a past period of anger.", "status": "opened", "clues": ["She had spent fifteen years being angry at him.", "She was not sure anymore what she was feeling."]}': 1,
    'Elena sat by the window.': 1,
    'Elena watched the rain.': 1,
    'Elena thought about her father.': 1,
    'There is a photograph of Elena\'s father.': 1,
    "Elena's father in the photograph appears younger and smiles.": 1,
    'Elena has an old ashtray on her window sill.': 1,
    'The rain did not stop outside.': 1,
}

In [18]:
# ═══════════════════════════════════════════════════════════════
# TEST 4 — ACCUMULATION / BIBLE COHERENCE
# Plot row count growth curve across all 8 scenes
# Healthy shape: steep early scenes, flattening by scene 5-6
# Red flag: linear growth forever = redundancy not being caught
# Also checks whether any ambiguities were surfaced to user
# ═══════════════════════════════════════════════════════════════

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

df = get_bible_facts(PROJECT_ID)
scenes_df = get_scenes(PROJECT_ID).sort_values("scene_number")

cumulative_counts = []
running_total = 0

for _, scene_row in scenes_df.iterrows():
    facts_from_scene = df[df["source_scene_id"] == scene_row["id"]]
    running_total += len(facts_from_scene)
    label = next(
        (s["label"] for s in TEST_SCENES if s["scene_number"] == scene_row["scene_number"]),
        "unknown"
    )
    cumulative_counts.append({
        "scene_number": scene_row["scene_number"],
        "label": label,
        "new_facts": len(facts_from_scene),
        "cumulative": running_total
    })

counts_df = pd.DataFrame(cumulative_counts)

print("── TEST 4: ACCUMULATION ──\n")
print(f"{'Scene':<8} {'Label':<28} {'New Facts':<12} {'Cumulative'}")
print("-" * 60)
for _, row in counts_df.iterrows():
    bar = "█" * int(row["new_facts"])
    print(f"  {int(row['scene_number']):<6} {row['label']:<28} {bar:<12} {int(row['cumulative'])}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(counts_df["scene_number"], counts_df["cumulative"],
             marker='o', linewidth=2, color='steelblue')
axes[0].set_xlabel("Scene Number")
axes[0].set_ylabel("Cumulative Bible Facts")
axes[0].set_title("Bible Growth Curve\n(should flatten as story matures)")
axes[0].grid(True, alpha=0.3)

axes[1].bar(counts_df["scene_number"], counts_df["new_facts"], color='steelblue', alpha=0.7)
axes[1].set_xlabel("Scene Number")
axes[1].set_ylabel("New Facts Added")
axes[1].set_title("New Facts Per Scene\n(should decrease over time)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("accumulation_curve.png", dpi=100)
plt.show()
print("Saved: accumulation_curve.png")

# Check ambiguities surfaced
questions_df = pd.DataFrame(
    supabase.table("bible_questions")
    .select("*")
    .eq("project_id", PROJECT_ID)
    .execute().data
)
print(f"\nAmbiguities surfaced to user: {len(questions_df)}")
if len(questions_df) > 0:
    for _, row in questions_df.iterrows():
        print(f"  Scene {row['source_scene_id']}: {row['questions']}")

# Verdict
last_three_avg = counts_df["new_facts"].tail(3).mean()
first_three_avg = counts_df["new_facts"].head(3).mean()
flattening = last_three_avg < first_three_avg

print(f"\nAvg new facts (scenes 1-3): {first_three_avg:.1f}")
print(f"Avg new facts (scenes 6-8): {last_three_avg:.1f}")
print(f"Curve flattening: {'yes' if flattening else 'no — possible redundancy problem'}")
print(f"TARGET: later scenes add fewer facts than earlier scenes")
print(f"RESULT: {'✓ PASS' if flattening else '✗ FAIL — bible growing at same rate, check redundancy'}")

I0521 12:00:29.716217 24557447 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0521 12:00:29.716728 24557447 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0521 12:00:29.716740 24557447 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0521 12:00:29.716743 24557447 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0521 12:00:29.716746 24557447 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0521 12:00:29.716749 24557447 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0521 12:00:29.716752 24557447 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0521 12:00:29.716755 24557447 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0521 12:00:29.716758 24557447 ev_poll_posix.cc:593] FD from fork parent still i

── TEST 4: ACCUMULATION ──

Scene    Label                        New Facts    Cumulative
------------------------------------------------------------
  1      opening_intro                █████████████ 13
  2      bar_scene                    █████████    22
  3      metro_implicit               ███████████  33
  4      three_characters             ██████████████████████ 55
  5      subtext_only                 ████████     63
  6      continuity_test              ████████     71
  7      world_building_dense         █████████    80
  8      redundancy_test              █████████████████████ 101
Saved: accumulation_curve.png

Ambiguities surfaced to user: 3
  Scene c9345c96-7bbc-463b-859c-a592320da253: ['The specific purpose or nature of the dojo.', 'The cause of the dojo fire.', 'The identity or meaning of the symbol on the matchbook.', "The origin story of Elena's scar.", 'The reasons why Prague smells like smoke more often in 2041.']
  Scene e5feaed5-cf49-41f4-9d6c-81abcb90c6c6: ["

/var/folders/1f/y4s45pzx3yx7z19rb57_y59w0000gn/T/ipykernel_74804/3350984635.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
# ═══════════════════════════════════════════════════════════════
# TEST 5 — BIBLE USAGE IN GENERATION
# Same description sent twice: once with full bible, once without
# Check if with-bible output references specific stored facts
# A bible that is being ignored = generation quality is no better
# than raw ChatGPT
# ═══════════════════════════════════════════════════════════════

import re

current_bible = get_bible_facts(PROJECT_ID).to_dict("records")
last_scene = TEST_SCENES[-1]["content"]

test_description = """Elena meets Vera again near the raised platforms in the lower 
district. Vera has new information about the photograph. Elena walks there because 
she has no other way to get around. It is raining."""

print("── TEST 5: BIBLE USAGE ──")
print(f"Bible size going into generation: {len(current_bible)} facts")
print(f"\nTest description:\n{test_description.strip()}\n")

print("Generating WITH bible...")
with_bible = generate_scene(test_description, current_bible, last_scene)
time.sleep(3)

print("Generating WITHOUT bible...")
without_bible = generate_scene(test_description, [], last_scene)

print("\n" + "="*60)
print("WITH BIBLE OUTPUT:")
print("="*60)
print(with_bible)

print("\n" + "="*60)
print("WITHOUT BIBLE OUTPUT:")
print("="*60)
print(without_bible)

# Automated checklist
bible_checks = [
    (r"platform|raised|elevated",        "raised platforms (world fact scene 7)"),
    (r"cannot drive|walked|on foot|walking|no car", "Elena cannot drive (scene 1)"),
    (r"\bVera\b",                         "Vera exists (scene 4)"),
    (r"Kova Corp|Kova",                   "Kova Corp (scene 2)"),
    (r"rain|wet|damp|drizzle",            "rain as recurring motif"),
    (r"left hand|scar|palm",              "scar on left hand (scene 1)"),
    (r"Zizkov|Prague|Czech",              "Prague setting (scene 1)"),
]

print("\n── CHECKLIST: bible facts in WITH-BIBLE output ──")
found_count = 0
for pattern, description in bible_checks:
    in_with = bool(re.search(pattern, with_bible, re.IGNORECASE))
    in_without = bool(re.search(pattern, without_bible, re.IGNORECASE))
    marker = "✓" if in_with else "✗"
    exclusive = "  ← only in with-bible" if in_with and not in_without else ""
    print(f"  {marker} {description}{exclusive}")
    if in_with:
        found_count += 1

print(f"\nBible facts referenced: {found_count}/{len(bible_checks)}")
print(f"TARGET: 4+ facts referenced, with-bible meaningfully different from without")
print(f"RESULT: {'✓ PASS' if found_count >= 4 else '✗ FAIL — generation ignoring bible, strengthen system prompt'}")

── TEST 5: BIBLE USAGE ──
Bible size going into generation: 101 facts

Test description:
Elena meets Vera again near the raised platforms in the lower 
district. Vera has new information about the photograph. Elena walks there because 
she has no other way to get around. It is raining.

Generating WITH bible...
Generating WITHOUT bible...

WITH BIBLE OUTPUT:
The rain-soaked streets of Holesovice gleamed like polished stone, the raised platforms a testament to Kova Corp's reconstruction efforts. Elena walked, her boots splashing in the puddles that collected in the low-lying areas. She had no other way to get around, no car to shield her from the elements. The rain drummed against her jacket, weighing it down, and her left hand, still wrapped in the makeshift bandage, throbbed with each step. She had been walking for what felt like hours, the cold air seeping into her bones, but she knew it was only a fraction of that time. As she turned a corner, she spotted Vera, standing near the edg

In [21]:
# ═══════════════════════════════════════════════════════════════
# TEST 7 — IMPLICIT FACTS
# Scene 3 (metro_implicit) contains facts shown through behavior
# not stated directly. Tests whether extraction goes beyond
# surface-level reading.
# Explicit facts: should always be caught
# Implicit facts: inferred from shown behavior — harder
# ═══════════════════════════════════════════════════════════════

metro_extracted = pipeline_results["metro_implicit"]
metro_flat = facts_to_flat_string(metro_extracted)

print("── TEST 7: IMPLICIT FACTS ──")
print("Scene: metro_implicit\n")
print("Full extracted output:")
print(json.dumps(metro_extracted, indent=2))

print("\n── EXPLICIT FACTS (stated directly — should all be caught) ──")
explicit_checks = [
    (r"metro|subway|transit",                "Elena takes the metro"),
    (r"midnight|late|past midnight|night",   "metro runs past midnight"),
]
explicit_found = 0
for pattern, description in explicit_checks:
    found = bool(re.search(pattern, metro_flat, re.IGNORECASE))
    print(f"  {'✓' if found else '✗'} {description}")
    if found:
        explicit_found += 1

print("\n── IMPLICIT FACTS (shown through behavior — harder to catch) ──")
implicit_checks = [
    (r"coin|cash|card|cautious|paranoi|habit|careful", "pays cash late at night (security habit)"),
    (r"backward|facing back|rear|last car",             "always sits facing backward"),
    (r"unknown|ignore|ring|not answer|let it",          "ignores unknown callers"),
    (r"cautious|habit|careful|old habit|instinct",      "has old cautious habits"),
]
implicit_found = 0
for pattern, description in implicit_checks:
    found = bool(re.search(pattern, metro_flat, re.IGNORECASE))
    print(f"  {'✓' if found else '✗'} {description}")
    if found:
        implicit_found += 1

print(f"\nExplicit recall: {explicit_found}/{len(explicit_checks)}")
print(f"Implicit recall: {implicit_found}/{len(implicit_checks)}")
print(f"\nTARGET: all explicit caught ({len(explicit_checks)}/{len(explicit_checks)}), at least 2 implicit caught")
explicit_pass = explicit_found == len(explicit_checks)
implicit_pass = implicit_found >= 2
print(f"Explicit: {'✓ PASS' if explicit_pass else '✗ FAIL'}")
print(f"Implicit: {'✓ PASS' if implicit_pass else '✗ FAIL (consider adding implicit fact instruction to extraction prompt)'}")
print(f"OVERALL: {'✓ PASS' if explicit_pass and implicit_pass else '✗ FAIL'}")

── TEST 7: IMPLICIT FACTS ──
Scene: metro_implicit

Full extracted output:
{
  "new_characters": [],
  "character_updates": [
    {
      "name": "Elena",
      "new_info": "Has a habit of paying for metro tickets with coins late at night instead of her card. Always sits in the last car of the metro, facing backward. Avoids answering calls from unknown numbers."
    }
  ],
  "world_facts": [
    "The Prague metro system operates late at night."
  ],
  "plot_threads": [
    {
      "thread": "Investigation into who is attempting to contact Elena.",
      "status": "opened",
      "clues": [
        "Elena received a phone call from an unknown number."
      ]
    }
  ],
  "established_facts": [
    "Elena bought a metro ticket with coins.",
    "Elena never uses her card for metro tickets late at night.",
    "Elena sat in the last car of the metro.",
    "Elena always faces backward in the metro.",
    "Elena's phone buzzed with an incoming call.",
    "The incoming call was from an un

In [22]:
# ═══════════════════════════════════════════════════════════════
# TEST 8 — TONE CONSISTENCY
# Generate 3 scenes at different points in the story
# using the accumulated bible as context
# Measure: sentence length, weather words, positive vs noir vocabulary
# Noir target profile:
#   avg sentence length 12-18 words
#   at least 1 weather reference
#   0-1 positive emotion words
#   3+ noir atmosphere words
# Variance across 3 scenes should be low (<5 words avg length diff)
# ═══════════════════════════════════════════════════════════════

current_bible = get_bible_facts(PROJECT_ID).to_dict("records")

tone_tests = [
    {
        "label": "early_tone",
        "description": "Elena searches the burned dojo for more clues. She finds a hidden trapdoor.",
        "prior_scene": TEST_SCENES[0]["content"]
    },
    {
        "label": "mid_tone",
        "description": "Tomas tells Elena that Vera has disappeared. Elena decides she needs to find the Mechanic.",
        "prior_scene": TEST_SCENES[3]["content"]
    },
    {
        "label": "late_tone",
        "description": "Elena confronts the boy about who gave him the photograph. He is more frightened than she expected.",
        "prior_scene": TEST_SCENES[7]["content"]
    },
]

def tone_signals(text: str) -> dict:
    sentences = [s.strip() for s in text.split('.') if len(s.strip()) > 5]
    avg_len = sum(len(s.split()) for s in sentences) / max(len(sentences), 1)
    weather = len(re.findall(
        r'\b(rain|fog|cold|grey|gray|dark|wet|damp|snow|cloud|smoke|ash|mist|drizzle)\b',
        text, re.I))
    positive = len(re.findall(
        r'\b(happy|smiled|laughed|warm|bright|wonderful|cheerful|joy|hope|light|glad|pleased)\b',
        text, re.I))
    noir = len(re.findall(
        r'\b(dark|cold|silence|alone|still|empty|nothing|dead|grey|shadow|quiet|hollow|numb)\b',
        text, re.I))
    return {
        "avg_sentence_length": round(avg_len, 1),
        "weather_words": weather,
        "positive_emotion": positive,
        "noir_atmosphere": noir,
        "sentence_count": len(sentences)
    }

print("── TEST 8: TONE CONSISTENCY ──\n")
tone_outputs = []

for i, test in enumerate(tone_tests):
    output = generate_scene(test["description"], current_bible, test["prior_scene"])
    signals = tone_signals(output)
    tone_outputs.append({
        "label": test["label"],
        "output": output,
        "signals": signals
    })

    len_ok = 12 <= signals["avg_sentence_length"] <= 22
    weather_ok = signals["weather_words"] >= 1
    positive_ok = signals["positive_emotion"] <= 2
    noir_ok = signals["noir_atmosphere"] >= 2

    print(f"Scene: {test['label']}")
    print(f"  {'✓' if len_ok else '✗'} Avg sentence length:  {signals['avg_sentence_length']} words  (target: 12-22)")
    print(f"  {'✓' if weather_ok else '✗'} Weather words:        {signals['weather_words']}             (target: 1+)")
    print(f"  {'✓' if positive_ok else '✗'} Positive emotion:     {signals['positive_emotion']}             (target: 0-2)")
    print(f"  {'✓' if noir_ok else '✗'} Noir atmosphere:      {signals['noir_atmosphere']}             (target: 2+)")
    print(f"\n  Text:\n  {output[:300]}...\n")

    if i < len(tone_tests) - 1:
        time.sleep(3)

avg_lengths = [t["signals"]["avg_sentence_length"] for t in tone_outputs]
variance = max(avg_lengths) - min(avg_lengths)

print(f"── CONSISTENCY CHECK ──")
print(f"Sentence length across 3 scenes: {avg_lengths}")
print(f"Variance: {variance:.1f} words  (target: <6)")
print(f"TARGET: consistent noir profile across all 3 scenes")
print(f"RESULT: {'✓ PASS' if variance < 6 else '✗ FAIL — tone drifting, add tone anchor to generation prompt'}")

── TEST 8: TONE CONSISTENCY ──

Scene: early_tone
  ✓ Avg sentence length:  19.0 words  (target: 12-22)
  ✓ Weather words:        5             (target: 1+)
  ✓ Positive emotion:     0             (target: 0-2)
  ✓ Noir atmosphere:      4             (target: 2+)

  Text:
  As Elena stepped into the charred remains of the dojo, the smell of smoke and ash enveloped her, a morbid familiarity in the cold Prague air. She navigated through the darkness, her eyes adjusting to the faint moonlight that filtered through the broken windows. The sound of her boots crunching on th...

Scene: mid_tone
  ✓ Avg sentence length:  18.9 words  (target: 12-22)
  ✓ Weather words:        3             (target: 1+)
  ✓ Positive emotion:     0             (target: 0-2)
  ✓ Noir atmosphere:      5             (target: 2+)

  Text:
  The dim lighting of Cerny Kohout seemed to swallow all sound, leaving only the soft hum of hushed conversations and the occasional clink of glasses. Tomas's voice, low and measure

In [23]:
# ═══════════════════════════════════════════════════════════════
# TEST 9 — CROSS-SCENE CONTINUITY
# Fact planted in scene 1: scar on LEFT hand
# Scene 6 explicitly confirms: Elena reopened the LEFT hand wound
# Two checks:
#   A — did bible preserve "left hand" from scene 1?
#   B — does generation use "left hand" when writing a hand injury?
# This tests the full chain: extraction → bible storage → generation use
# ═══════════════════════════════════════════════════════════════

print("── TEST 9: CROSS-SCENE CONTINUITY ──\n")

df = get_bible_facts(PROJECT_ID)

# Check A — is left hand in the bible at all?
bible_strings = df["content"].apply(content_to_string)
left_hand_rows = df[bible_strings.str.contains("left", case=False)]
passed_a = len(left_hand_rows) > 0

print("Check A — was 'left hand' preserved in bible from scene 1?")
print(f"  Result: {'✓ YES' if passed_a else '✗ NO — extraction missed the scar location'}")
if passed_a:
    for _, row in left_hand_rows.iterrows():
        print(f"  Found in [{row['category']}]: {content_to_string(row['content'])[:100]}")

# Check B — does generation use left hand in a hand injury scene?
print("\nCheck B — does generation correctly use 'left hand' for injury scene?")
print("  Generating scene with hand injury description...\n")

continuity_description = """Elena cuts her hand badly on a broken window frame while 
escaping through a building. She has to bandage it herself in a stairwell. 
It is the same hand as always."""

current_bible = get_bible_facts(PROJECT_ID).to_dict("records")
continuity_output = generate_scene(
    continuity_description,
    current_bible,
    TEST_SCENES[4]["content"]
)

print("Generated scene:")
print(continuity_output)

left_mentioned = bool(re.search(r'\bleft\b', continuity_output, re.IGNORECASE))
hand_mentioned = bool(re.search(r'\bhand\b', continuity_output, re.IGNORECASE))
scar_mentioned = bool(re.search(r'\bscar\b', continuity_output, re.IGNORECASE))
passed_b = left_mentioned and hand_mentioned

print(f"\nCheck B results:")
print(f"  {'✓' if left_mentioned else '✗'} 'left' mentioned in output")
print(f"  {'✓' if hand_mentioned else '✗'} 'hand' mentioned in output")
print(f"  {'✓' if scar_mentioned else '✗'} 'scar' mentioned (bonus — shows deep bible use)")

print(f"\nCheck A (bible preserved the fact):   {'✓ PASS' if passed_a else '✗ FAIL'}")
print(f"Check B (generation used the fact):   {'✓ PASS' if passed_b else '✗ FAIL'}")
print(f"\nTARGET: both A and B pass")
print(f"OVERALL: {'✓ PASS' if passed_a and passed_b else '✗ FAIL — continuity chain has a gap'}")

if not passed_a:
    print("\nFix: strengthen extraction prompt to capture physical details explicitly")
if passed_a and not passed_b:
    print("\nFix: generation is not reading bible carefully enough — strengthen system prompt")

── TEST 9: CROSS-SCENE CONTINUITY ──

Check A — was 'left hand' preserved in bible from scene 1?
  Result: ✓ YES
  Found in [character]: elena main character red hair, has a pale and old scar running across her left palm does not talk ab
  Found in [character]: elena elena's old scar tissue on her left palm was reopened.
  Found in [established_fact]: elena picked up the photograph with her left hand.

Check B — does generation correctly use 'left hand' for injury scene?
  Generating scene with hand injury description...

Generated scene:
The darkness of the building enveloped her, a damp shroud that clung to her skin as she moved. Elena's eyes adjusted slowly, her breathing quiet and controlled as she navigated the deserted corridors. She had to keep moving, had to find a way out. A broken window frame loomed before her, a jagged mouth of splintered wood and shattered glass. Without hesitation, she reached out to climb through, her left hand closing around the frame as she hoisted her

In [24]:
# ═══════════════════════════════════════════════════════════════
# FULL ITERATION SUMMARY
# Fill in each result after running the test above it
# Use this to track what changed between prompt iterations
# ═══════════════════════════════════════════════════════════════

ITERATION_NUMBER = 1

print(f"╔══════════════════════════════════════════════════════════════╗")
print(f"║         ITERATION {ITERATION_NUMBER} — FULL TEST RESULTS SUMMARY           ║")
print(f"╠══════════════════════════════════════════════════════════════╣")

results = [
    ("6", "JSON Reliability",        "___/8  (target 8/8)       "),
    ("1", "Extraction Recall",        "___%   (target 80%+)      "),
    ("2", "Redundancy",               "___ pairs (target 0)      "),
    ("3", "Hallucination",            "___ invented (target 0)   "),
    ("4", "Accumulation",             "flattens? Y/N             "),
    ("5", "Bible Usage",              "___/7  (target 4+)        "),
    ("7", "Implicit Facts",           "___/4  (target 2+)        "),
    ("8", "Tone Consistency",         "variance ___w (target <6) "),
    ("9", "Cross-scene Continuity",   "A: Y/N   B: Y/N           "),
]

for num, name, result in results:
    print(f"║  Test {num}: {name:<28} {result}║")

print(f"╠══════════════════════════════════════════════════════════════╣")
print(f"║  NOTES                                                       ║")
print(f"╚══════════════════════════════════════════════════════════════╝")

print(f"""
Iteration {ITERATION_NUMBER} notes:
  Extraction prompt: [describe version]
  Generation prompt: [describe version]
  
  What failed:
  - 
  
  What to change for iteration {ITERATION_NUMBER + 1}:
  - (change ONE thing at a time)
  
  To re-run: 
    1. Edit prompt variables in the Prompts cell
    2. Re-run Prompts cell + Functions cell
    3. Run Setup cell (clears DB, re-runs all 8 scenes)
    4. Run tests 1-9 in order
    5. Update ITERATION_NUMBER above and fill in results
""")

╔══════════════════════════════════════════════════════════════╗
║         ITERATION 1 — FULL TEST RESULTS SUMMARY           ║
╠══════════════════════════════════════════════════════════════╣
║  Test 6: JSON Reliability             ___/8  (target 8/8)       ║
║  Test 1: Extraction Recall            ___%   (target 80%+)      ║
║  Test 2: Redundancy                   ___ pairs (target 0)      ║
║  Test 3: Hallucination                ___ invented (target 0)   ║
║  Test 4: Accumulation                 flattens? Y/N             ║
║  Test 5: Bible Usage                  ___/7  (target 4+)        ║
║  Test 7: Implicit Facts               ___/4  (target 2+)        ║
║  Test 8: Tone Consistency             variance ___w (target <6) ║
║  Test 9: Cross-scene Continuity       A: Y/N   B: Y/N           ║
╠══════════════════════════════════════════════════════════════╣
║  NOTES                                                       ║
╚══════════════════════════════════════════════════════════════╝



## Iteration 2

In [27]:
# ═══════════════════════════════════════════════════════════════
# SETUP — clear DB and run all 8 scenes through full pipeline
# Gemini free tier: 5 req/min → 12s minimum between calls
# With retry logic for quota errors
# ═══════════════════════════════════════════════════════════════

import re as _re
import time

def extract_facts_with_retry(scene_content, existing_facts, max_retries=3):
    """Wraps extract_facts with retry logic for rate limit errors."""
    for attempt in range(max_retries):
        try:
            return extract_facts(scene_content, existing_facts)
        except Exception as e:
            err = str(e)
            if "429" in err or "RESOURCE_EXHAUSTED" in err or "quota" in err.lower():
                # Try to parse wait time from error message
                match = _re.search(r'retry in (\d+)', err)
                wait = int(match.group(1)) + 3 if match else 35
                print(f"  Rate limited. Waiting {wait}s before retry {attempt+1}/{max_retries}...")
                time.sleep(wait)
            else:
                raise  # non-rate-limit error, don't retry
    raise Exception("Max retries exceeded")

def run_pipeline_safe(project_id, scene_number, scene_content, save_to_supabase=True):
    """Same as run_pipeline but uses retry-aware extraction."""
    existing = supabase.table("bible_facts")\
        .select("*")\
        .eq("project_id", project_id)\
        .eq("status", "confirmed")\
        .execute()
    existing_facts = existing.data or []
    print(f"  Existing bible: {len(existing_facts)} facts")

    extracted = extract_facts_with_retry(scene_content, existing_facts)
    print(f"  Extracted: {json.dumps(extracted, indent=2)}")

    if save_to_supabase:
        scene_result = supabase.table("scenes").insert({
            "project_id": project_id,
            "scene_number": scene_number,
            "version": 1,
            "content": scene_content,
            "status": "active"
        }).execute()
        scene_id = scene_result.data[0]["id"]

        rows = []
        for char in extracted.get("new_characters", []):
            rows.append({"project_id": project_id, "source_scene_id": scene_id,
                        "category": "character", "content": char, "status": "confirmed"})
        for update in extracted.get("character_updates", []):
            rows.append({"project_id": project_id, "source_scene_id": scene_id,
                        "category": "character", "content": update, "status": "confirmed"})
        for fact in extracted.get("world_facts", []):
            rows.append({"project_id": project_id, "source_scene_id": scene_id,
                        "category": "world", "content": {"fact": fact}, "status": "confirmed"})
        for thread in extracted.get("plot_threads", []):
            rows.append({"project_id": project_id, "source_scene_id": scene_id,
                        "category": "plot_thread", "content": thread, "status": "confirmed"})
        for fact in extracted.get("established_facts", []):
            rows.append({"project_id": project_id, "source_scene_id": scene_id,
                        "category": "established_fact", "content": {"fact": fact}, "status": "confirmed"})

        if rows:
            supabase.table("bible_facts").insert(rows).execute()

        if extracted.get("ambiguities"):
            supabase.table("bible_questions").insert({
                "project_id": project_id,
                "source_scene_id": scene_id,
                "questions": extracted["ambiguities"],
                "status": "pending"
            }).execute()

        print(f"  Saved {len(rows)} facts to Supabase")

    return extracted

# ── MAIN SETUP RUN ───────────────────────────────────────────
RATE_LIMIT_DELAY = 13  # 12s minimum + 1s buffer for 5 req/min

clear_project_data(PROJECT_ID)
print("DB cleared\n")

pipeline_results = {}

for i, scene in enumerate(TEST_SCENES):
    print(f"Scene {scene['scene_number']}/{len(TEST_SCENES)}: {scene['label']}")
    extracted = run_pipeline_safe(PROJECT_ID, scene["scene_number"], scene["content"])
    pipeline_results[scene["label"]] = extracted

    if i < len(TEST_SCENES) - 1:
        print(f"  waiting {RATE_LIMIT_DELAY}s...\n")
        time.sleep(RATE_LIMIT_DELAY)

print(f"\n{'='*50}")
print(f"All {len(TEST_SCENES)} scenes processed")
print(f"pipeline_results keys: {list(pipeline_results.keys())}")
print(f"Total expected runtime: ~{len(TEST_SCENES) * RATE_LIMIT_DELAY}s")

Cleared all data for project 73a1bada-b2bf-423b-8c6e-f107076da2ed
DB cleared

Scene 1/8: opening_intro
  Existing bible: 0 facts
  Extracted: {
  "new_characters": [
    {
      "name": "Elena",
      "physical": "Red hair; has an old, pale scar across her left palm.",
      "personality": null,
      "role": "Protagonist (implied)",
      "relationships": null
    }
  ],
  "character_updates": [],
  "world_facts": [
    "The current year is 2041.",
    "Prague frequently smells like smoke.",
    "The frequency of smoke in Prague has increased compared to the past.",
    "There is a district in Prague called Zizkov.",
    "There is a dojo in Prague."
  ],
  "plot_threads": [
    {
      "thread": "Investigate the circumstances of the dojo building being smoldering.",
      "status": "opened",
      "clues": [
        "The dojo building was still smoldering just after midnight."
      ]
    },
    {
      "thread": "Uncover the meaning and origin of the matchbook and its recognized symb

In [28]:
# ═══════════════════════════════════════════════════════════════
# TEST 6 — JSON RELIABILITY
# One extraction per scene (8 total) with rate limit respect
# Gemini free tier: 10 requests/min → 6s minimum between calls
# ═══════════════════════════════════════════════════════════════

import time

json_results = []
RATE_LIMIT_DELAY = 6  # seconds — keeps us under 10 req/min

print("Running 8 extractions for JSON reliability (6s delay between calls)...\n")

for i, scene in enumerate(TEST_SCENES):
    try:
        result = extract_facts(scene["content"], [])
        json_results.append({
            "scene": scene["label"], 
            "run": i+1, 
            "status": "pass", 
            "error": None
        })
        print(f"  Run {i+1:02d} [{scene['label']}]: PASS")
    except json.JSONDecodeError as e:
        json_results.append({
            "scene": scene["label"], 
            "run": i+1, 
            "status": "fail", 
            "error": f"JSON parse error: {str(e)[:80]}"
        })
        print(f"  Run {i+1:02d} [{scene['label']}]: FAIL (JSON) — {str(e)[:60]}")
    except Exception as e:
        json_results.append({
            "scene": scene["label"], 
            "run": i+1, 
            "status": "fail", 
            "error": str(e)[:80]
        })
        print(f"  Run {i+1:02d} [{scene['label']}]: FAIL (other) — {str(e)[:60]}")
    
    if i < len(TEST_SCENES) - 1:  # no delay after last one
        print(f"         waiting {RATE_LIMIT_DELAY}s...")
        time.sleep(RATE_LIMIT_DELAY)

passed = sum(1 for r in json_results if r["status"] == "pass")
json_failures = [r for r in json_results if r["status"] == "fail" 
                 and "JSON" in r.get("error", "")]
other_failures = [r for r in json_results if r["status"] == "fail" 
                  and "JSON" not in r.get("error", "")]

print(f"\n── JSON RELIABILITY RESULT ──")
print(f"Passed:            {passed}/8")
print(f"JSON parse errors: {len(json_failures)}  ← actual reliability failures")
print(f"Other errors:      {len(other_failures)}  ← rate limits, network, etc (not counted)")

if len(json_failures) == 0:
    print("✓ PASS — JSON is reliable, safe to proceed")
else:
    print("✗ FAIL — JSON parsing unreliable, fix extraction prompt")
    for f in json_failures:
        print(f"  Failed: {f['scene']} — {f['error']}")

Running 8 extractions for JSON reliability (6s delay between calls)...

  Run 01 [opening_intro]: PASS
         waiting 6s...
  Run 02 [bar_scene]: PASS
         waiting 6s...
  Run 03 [metro_implicit]: PASS
         waiting 6s...
  Run 04 [three_characters]: FAIL (other) — 429 You exceeded your current quota, please check your plan 
         waiting 6s...
  Run 05 [subtext_only]: FAIL (other) — 429 You exceeded your current quota, please check your plan 
         waiting 6s...
  Run 06 [continuity_test]: FAIL (other) — 429 You exceeded your current quota, please check your plan 
         waiting 6s...
  Run 07 [world_building_dense]: FAIL (other) — 429 You exceeded your current quota, please check your plan 
         waiting 6s...
  Run 08 [redundancy_test]: FAIL (other) — 429 You exceeded your current quota, please check your plan 

── JSON RELIABILITY RESULT ──
Passed:            3/8
JSON parse errors: 0  ← actual reliability failures
Other errors:      5  ← rate limits, network, et

In [29]:
# ═══════════════════════════════════════════════════════════════
# TEST 1 — EXTRACTION RECALL
# For each scene tagged with test_target 1:
#   list planted facts, check which appeared in extracted output
#   score = found / planted per scene, then overall
# Word overlap >= 0.5 counts as "found"
# ═══════════════════════════════════════════════════════════════

def facts_to_flat_string(extracted: dict) -> str:
    parts = []
    for key, val in extracted.items():
        if isinstance(val, list):
            for item in val:
                if isinstance(item, dict):
                    parts.append(json.dumps(item).lower())
                else:
                    parts.append(str(item).lower())
    return " ".join(parts)

def word_overlap(a: str, b: str) -> float:
    words_a = set(a.lower().split())
    words_b = set(b.lower().split())
    if not words_a:
        return 0.0
    return len(words_a & words_b) / len(words_a)

recall_results = []
recall_scenes = [s for s in TEST_SCENES if 1 in s["test_targets"]]

for scene in recall_scenes:
    extracted_flat = facts_to_flat_string(pipeline_results[scene["label"]])
    found = []
    missed = []
    for fact in scene["planted_facts"]:
        overlap = word_overlap(fact, extracted_flat)
        if overlap >= 0.5:
            found.append(fact)
        else:
            missed.append(fact)
    recall = len(found) / len(scene["planted_facts"])
    recall_results.append({
        "scene": scene["label"],
        "planted": len(scene["planted_facts"]),
        "found": len(found),
        "missed": missed,
        "recall": recall
    })

print("── TEST 1: EXTRACTION RECALL ──\n")
total_planted = sum(r["planted"] for r in recall_results)
total_found = sum(r["found"] for r in recall_results)

for r in recall_results:
    status = "✓" if r["recall"] >= 0.8 else "✗"
    print(f"{status} {r['scene']:<25} {r['found']}/{r['planted']} ({r['recall']*100:.0f}%)")
    if r["missed"]:
        for m in r["missed"]:
            print(f"    MISSED: {m}")

overall = total_found / total_planted
print(f"\nOVERALL RECALL: {total_found}/{total_planted} ({overall*100:.0f}%)")
print(f"TARGET: 80%+")
print(f"RESULT: {'✓ PASS' if overall >= 0.8 else '✗ FAIL — improve extraction prompt'}")

── TEST 1: EXTRACTION RECALL ──

✓ opening_intro             6/6 (100%)
✓ bar_scene                 5/5 (100%)
✓ metro_implicit            5/6 (83%)
    MISSED: Elena pays with coins late at night, never card
✗ three_characters          5/8 (62%)
    MISSED: The Mechanic is heavyset and says nothing
    MISSED: The Mechanic watches exits
    MISSED: The boy wears a Kova Corp badge upside down

OVERALL RECALL: 21/25 (84%)
TARGET: 80%+
RESULT: ✓ PASS


In [30]:
# ═══════════════════════════════════════════════════════════════
# TEST 2 — REDUNDANCY
# Compare every pair of facts within same category
# Flag pairs with >70% word overlap as likely duplicates
# Scene 7 and 8 are the main risk — they re-mention established facts
# ═══════════════════════════════════════════════════════════════

def content_to_string(content) -> str:
    if isinstance(content, dict):
        return " ".join(str(v) for v in content.values()).lower()
    return str(content).lower()

df = get_bible_facts(PROJECT_ID)
redundant_pairs = []

for category in ["character", "world", "plot_thread", "established_fact"]:
    cat_rows = df[df["category"] == category].copy()
    cat_rows["content_str"] = cat_rows["content"].apply(content_to_string)
    rows = cat_rows.to_dict("records")
    for i in range(len(rows)):
        for j in range(i+1, len(rows)):
            overlap = word_overlap(rows[i]["content_str"], rows[j]["content_str"])
            if overlap >= 0.7:
                redundant_pairs.append({
                    "category": category,
                    "overlap": round(overlap, 2),
                    "fact_a": rows[i]["content_str"][:120],
                    "fact_b": rows[j]["content_str"][:120],
                })

print("── TEST 2: REDUNDANCY ──\n")
print(f"Total bible facts in DB: {len(df)}")
print(f"Breakdown by category:")
for cat, count in df["category"].value_counts().items():
    print(f"  {cat:<20} {count} rows")

print(f"\nSuspected redundant pairs (>70% word overlap): {len(redundant_pairs)}")

if redundant_pairs:
    print("\nRedundant pairs:")
    for p in redundant_pairs:
        print(f"\n  [{p['category']}] overlap={p['overlap']}")
        print(f"  A: {p['fact_a']}")
        print(f"  B: {p['fact_b']}")
else:
    print("No redundant pairs found")

print(f"\nTARGET: 0 pairs above 70% overlap")
print(f"RESULT: {'✓ PASS' if len(redundant_pairs) == 0 else f'✗ FAIL — {len(redundant_pairs)} redundant pairs found'}")

── TEST 2: REDUNDANCY ──

Total bible facts in DB: 63
Breakdown by category:
  world                24 rows
  established_fact     23 rows
  character            11 rows
  plot_thread          5 rows

Suspected redundant pairs (>70% word overlap): 5

Redundant pairs:

  [world] overlap=0.75
  A: there is a location in prague called narodni.
  B: there is a district in prague called zizkov.

  [world] overlap=0.73
  A: the vltava river has flooded prague three times in the last decade.
  B: the severity of vltava river floods in prague has increased over the last decade.

  [world] overlap=0.83
  A: there is a dojo in prague.
  B: there is a bar called cerny kohout (the black rooster) in prague.

  [established_fact] overlap=0.75
  A: elena's father was quiet.
  B: elena's father was careful.

  [established_fact] overlap=0.86
  A: both men in the photograph are smiling.
  B: both men in the photograph are standing in front of the dojo.

TARGET: 0 pairs above 70% overlap
RESULT: ✗ FAIL 

In [31]:
# ═══════════════════════════════════════════════════════════════
# TEST 3 — HALLUCINATION (part 1 of 2)
# Scene 5 (subtext_only) has almost no extractable facts
# Only 2 facts were planted — everything else is a hallucination candidate
# Read the output carefully then fill in labels in the next cell
# ═══════════════════════════════════════════════════════════════

subtext_extracted = pipeline_results["subtext_only"]

print("── TEST 3: HALLUCINATION ──")
print("Scene: subtext_only")
print("\nOnly 2 real facts were planted in this scene:")
print("  1. Elena has not contacted her father in fifteen years")
print("  2. Elena stopped smoking three years ago")
print("\nFull extracted output:")
print(json.dumps(subtext_extracted, indent=2))

print("\n── ALL FACTS EXTRACTED FROM THIS SCENE ──")
print("Read each one and decide: is it grounded in the text, or invented?\n")

all_facts_from_subtext = []
for key in ["new_characters", "character_updates", "world_facts",
            "plot_threads", "established_facts"]:
    items = subtext_extracted.get(key, [])
    for item in items:
        fact_str = json.dumps(item) if isinstance(item, dict) else str(item)
        all_facts_from_subtext.append(fact_str)
        print(f"  [ ] {fact_str[:140]}")

print(f"\nTotal extracted: {len(all_facts_from_subtext)}")
print("\nNow fill in hallucination_labels in the next cell and run it")

── TEST 3: HALLUCINATION ──
Scene: subtext_only

Only 2 real facts were planted in this scene:
  1. Elena has not contacted her father in fifteen years
  2. Elena stopped smoking three years ago

Full extracted output:
{
  "new_characters": [],
  "character_updates": [
    {
      "name": "Elena",
      "new_info": "Has a father; was angry at her father for fifteen years; stopped smoking three years ago."
    },
    {
      "name": "Elena's Father",
      "new_info": "Was quiet and careful; habitually left rooms unnoticed."
    }
  ],
  "world_facts": [],
  "plot_threads": [],
  "established_facts": [
    "Elena has a father.",
    "Elena's father was quiet.",
    "Elena's father was careful.",
    "Elena's father habitually left rooms unnoticed.",
    "Elena was angry at her father for fifteen years.",
    "Elena stopped smoking three years ago."
  ],
  "ambiguities": [
    "The identity of 'the man in the photograph' of Elena's father, and why he is distinct from the man she remember

In [32]:
hallucination_labels = {
    '{"name": "Elena\'s Father", "physical": "appears younger and smiles in a photograph", "personality": "quiet, careful, always leaving a room slightly before anyone noticed (as remembered by Elena)", "role": "Elena\'s father", "relationships": "father of Elena"}': 1,
    '{"name": "Elena", "new_info": "Did not sleep."}': 1,
    '{"name": "Elena", "new_info": "Has a father she thinks about."}': 1,
    '{"name": "Elena", "new_info": "Spent fifteen years being angry at her father."}': 1,
    '{"name": "Elena", "new_info": "Is now unsure what she feels about her father."}': 1,
    '{"name": "Elena", "new_info": "Has not smoked in three years."}': 1,
    '{"thread": "Elena\'s complicated relationship with her father, including a past period of anger.", "status": "opened", "clues": ["She had spent fifteen years being angry at him.", "She was not sure anymore what she was feeling."]}': 1,
    'Elena sat by the window.': 1,
    'Elena watched the rain.': 1,
    'Elena thought about her father.': 1,
    'There is a photograph of Elena\'s father.': 1,
    "Elena's father in the photograph appears younger and smiles.": 1,
    'Elena has an old ashtray on her window sill.': 1,
    'The rain did not stop outside.': 1,
}

In [33]:
# ═══════════════════════════════════════════════════════════════
# TEST 4 — ACCUMULATION / BIBLE COHERENCE
# Plot row count growth curve across all 8 scenes
# Healthy shape: steep early scenes, flattening by scene 5-6
# Red flag: linear growth forever = redundancy not being caught
# Also checks whether any ambiguities were surfaced to user
# ═══════════════════════════════════════════════════════════════

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

df = get_bible_facts(PROJECT_ID)
scenes_df = get_scenes(PROJECT_ID).sort_values("scene_number")

cumulative_counts = []
running_total = 0

for _, scene_row in scenes_df.iterrows():
    facts_from_scene = df[df["source_scene_id"] == scene_row["id"]]
    running_total += len(facts_from_scene)
    label = next(
        (s["label"] for s in TEST_SCENES if s["scene_number"] == scene_row["scene_number"]),
        "unknown"
    )
    cumulative_counts.append({
        "scene_number": scene_row["scene_number"],
        "label": label,
        "new_facts": len(facts_from_scene),
        "cumulative": running_total
    })

counts_df = pd.DataFrame(cumulative_counts)

print("── TEST 4: ACCUMULATION ──\n")
print(f"{'Scene':<8} {'Label':<28} {'New Facts':<12} {'Cumulative'}")
print("-" * 60)
for _, row in counts_df.iterrows():
    bar = "█" * int(row["new_facts"])
    print(f"  {int(row['scene_number']):<6} {row['label']:<28} {bar:<12} {int(row['cumulative'])}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(counts_df["scene_number"], counts_df["cumulative"],
             marker='o', linewidth=2, color='steelblue')
axes[0].set_xlabel("Scene Number")
axes[0].set_ylabel("Cumulative Bible Facts")
axes[0].set_title("Bible Growth Curve\n(should flatten as story matures)")
axes[0].grid(True, alpha=0.3)

axes[1].bar(counts_df["scene_number"], counts_df["new_facts"], color='steelblue', alpha=0.7)
axes[1].set_xlabel("Scene Number")
axes[1].set_ylabel("New Facts Added")
axes[1].set_title("New Facts Per Scene\n(should decrease over time)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("accumulation_curve.png", dpi=100)
plt.show()
print("Saved: accumulation_curve.png")

# Check ambiguities surfaced
questions_df = pd.DataFrame(
    supabase.table("bible_questions")
    .select("*")
    .eq("project_id", PROJECT_ID)
    .execute().data
)
print(f"\nAmbiguities surfaced to user: {len(questions_df)}")
if len(questions_df) > 0:
    for _, row in questions_df.iterrows():
        print(f"  Scene {row['source_scene_id']}: {row['questions']}")

# Verdict
last_three_avg = counts_df["new_facts"].tail(3).mean()
first_three_avg = counts_df["new_facts"].head(3).mean()
flattening = last_three_avg < first_three_avg

print(f"\nAvg new facts (scenes 1-3): {first_three_avg:.1f}")
print(f"Avg new facts (scenes 6-8): {last_three_avg:.1f}")
print(f"Curve flattening: {'yes' if flattening else 'no — possible redundancy problem'}")
print(f"TARGET: later scenes add fewer facts than earlier scenes")
print(f"RESULT: {'✓ PASS' if flattening else '✗ FAIL — bible growing at same rate, check redundancy'}")

── TEST 4: ACCUMULATION ──

Scene    Label                        New Facts    Cumulative
------------------------------------------------------------
  1      opening_intro                ████████████ 12
  2      bar_scene                    ██████████   22
  3      metro_implicit               ████         26
  4      three_characters             ██████████   36
  5      subtext_only                 ████████     44
  6      continuity_test              █            45
  7      world_building_dense         █████████    54
  8      redundancy_test              █████████    63
Saved: accumulation_curve.png

Ambiguities surfaced to user: 4
  Scene 84db45de-e515-4706-97c2-47cf7798c742: ['What is the specific symbol embossed on the matchbook that Elena recognizes?', "What event caused the old, pale scar on Elena's left palm?"]
  Scene bc2251d5-4498-47c4-a744-50e9c5c0cbf8: ['The identity of the unknown number that called Elena.']
  Scene 72f69558-3172-419b-9f35-47c02a26d05f: ['The content a

/var/folders/1f/y4s45pzx3yx7z19rb57_y59w0000gn/T/ipykernel_74804/3350984635.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [34]:
# ═══════════════════════════════════════════════════════════════
# TEST 5 — BIBLE USAGE IN GENERATION
# Same description sent twice: once with full bible, once without
# Check if with-bible output references specific stored facts
# A bible that is being ignored = generation quality is no better
# than raw ChatGPT
# ═══════════════════════════════════════════════════════════════

import re

current_bible = get_bible_facts(PROJECT_ID).to_dict("records")
last_scene = TEST_SCENES[-1]["content"]

test_description = """Elena meets Vera again near the raised platforms in the lower 
district. Vera has new information about the photograph. Elena walks there because 
she has no other way to get around. It is raining."""

print("── TEST 5: BIBLE USAGE ──")
print(f"Bible size going into generation: {len(current_bible)} facts")
print(f"\nTest description:\n{test_description.strip()}\n")

print("Generating WITH bible...")
with_bible = generate_scene(test_description, current_bible, last_scene)
time.sleep(3)

print("Generating WITHOUT bible...")
without_bible = generate_scene(test_description, [], last_scene)

print("\n" + "="*60)
print("WITH BIBLE OUTPUT:")
print("="*60)
print(with_bible)

print("\n" + "="*60)
print("WITHOUT BIBLE OUTPUT:")
print("="*60)
print(without_bible)

# Automated checklist
bible_checks = [
    (r"platform|raised|elevated",        "raised platforms (world fact scene 7)"),
    (r"cannot drive|walked|on foot|walking|no car", "Elena cannot drive (scene 1)"),
    (r"\bVera\b",                         "Vera exists (scene 4)"),
    (r"Kova Corp|Kova",                   "Kova Corp (scene 2)"),
    (r"rain|wet|damp|drizzle",            "rain as recurring motif"),
    (r"left hand|scar|palm",              "scar on left hand (scene 1)"),
    (r"Zizkov|Prague|Czech",              "Prague setting (scene 1)"),
]

print("\n── CHECKLIST: bible facts in WITH-BIBLE output ──")
found_count = 0
for pattern, description in bible_checks:
    in_with = bool(re.search(pattern, with_bible, re.IGNORECASE))
    in_without = bool(re.search(pattern, without_bible, re.IGNORECASE))
    marker = "✓" if in_with else "✗"
    exclusive = "  ← only in with-bible" if in_with and not in_without else ""
    print(f"  {marker} {description}{exclusive}")
    if in_with:
        found_count += 1

print(f"\nBible facts referenced: {found_count}/{len(bible_checks)}")
print(f"TARGET: 4+ facts referenced, with-bible meaningfully different from without")
print(f"RESULT: {'✓ PASS' if found_count >= 4 else '✗ FAIL — generation ignoring bible, strengthen system prompt'}")

── TEST 5: BIBLE USAGE ──
Bible size going into generation: 63 facts

Test description:
Elena meets Vera again near the raised platforms in the lower 
district. Vera has new information about the photograph. Elena walks there because 
she has no other way to get around. It is raining.

Generating WITH bible...
Generating WITHOUT bible...

WITH BIBLE OUTPUT:
The rain-soaked streets of Prague's lower districts were always a dismal sight, but tonight they seemed particularly forlorn. Elena walked, her boots splashing in the puddles that reflected the faint glow of the streetlights. She had no other way to get around, no car to shield her from the constant drizzle that seemed to seep into every pore. The raised platforms that now supported the rebuilt districts of Holesovice, Karlin, and Zizkov loomed above her, a reminder of Kova Corp's influence on the city. It was near one of these platforms that she was supposed to meet Vera, the timid employee from Kova Corp's legal department.

As sh

In [35]:
# ═══════════════════════════════════════════════════════════════
# TEST 7 — IMPLICIT FACTS
# Scene 3 (metro_implicit) contains facts shown through behavior
# not stated directly. Tests whether extraction goes beyond
# surface-level reading.
# Explicit facts: should always be caught
# Implicit facts: inferred from shown behavior — harder
# ═══════════════════════════════════════════════════════════════

metro_extracted = pipeline_results["metro_implicit"]
metro_flat = facts_to_flat_string(metro_extracted)

print("── TEST 7: IMPLICIT FACTS ──")
print("Scene: metro_implicit\n")
print("Full extracted output:")
print(json.dumps(metro_extracted, indent=2))

print("\n── EXPLICIT FACTS (stated directly — should all be caught) ──")
explicit_checks = [
    (r"metro|subway|transit",                "Elena takes the metro"),
    (r"midnight|late|past midnight|night",   "metro runs past midnight"),
]
explicit_found = 0
for pattern, description in explicit_checks:
    found = bool(re.search(pattern, metro_flat, re.IGNORECASE))
    print(f"  {'✓' if found else '✗'} {description}")
    if found:
        explicit_found += 1

print("\n── IMPLICIT FACTS (shown through behavior — harder to catch) ──")
implicit_checks = [
    (r"coin|cash|card|cautious|paranoi|habit|careful", "pays cash late at night (security habit)"),
    (r"backward|facing back|rear|last car",             "always sits facing backward"),
    (r"unknown|ignore|ring|not answer|let it",          "ignores unknown callers"),
    (r"cautious|habit|careful|old habit|instinct",      "has old cautious habits"),
]
implicit_found = 0
for pattern, description in implicit_checks:
    found = bool(re.search(pattern, metro_flat, re.IGNORECASE))
    print(f"  {'✓' if found else '✗'} {description}")
    if found:
        implicit_found += 1

print(f"\nExplicit recall: {explicit_found}/{len(explicit_checks)}")
print(f"Implicit recall: {implicit_found}/{len(implicit_checks)}")
print(f"\nTARGET: all explicit caught ({len(explicit_checks)}/{len(explicit_checks)}), at least 2 implicit caught")
explicit_pass = explicit_found == len(explicit_checks)
implicit_pass = implicit_found >= 2
print(f"Explicit: {'✓ PASS' if explicit_pass else '✗ FAIL'}")
print(f"Implicit: {'✓ PASS' if implicit_pass else '✗ FAIL (consider adding implicit fact instruction to extraction prompt)'}")
print(f"OVERALL: {'✓ PASS' if explicit_pass and implicit_pass else '✗ FAIL'}")

── TEST 7: IMPLICIT FACTS ──
Scene: metro_implicit

Full extracted output:
{
  "new_characters": [],
  "character_updates": [
    {
      "name": "Elena",
      "new_info": "Prefers not to use her card for transactions late at night; has old habits from past experiences she prefers not to think about; habitually sits in the last car of the metro; habitually faces backward when riding the metro; perceives the city as quieter and more 'honest' at night."
    }
  ],
  "world_facts": [
    "The metro system in Prague operates at night.",
    "Physical money (coins) is a valid form of payment for metro tickets in Prague."
  ],
  "plot_threads": [],
  "established_facts": [
    "Elena owns a phone."
  ],
  "ambiguities": [
    "The identity of the unknown number that called Elena."
  ]
}

── EXPLICIT FACTS (stated directly — should all be caught) ──
  ✓ Elena takes the metro
  ✓ metro runs past midnight

── IMPLICIT FACTS (shown through behavior — harder to catch) ──
  ✓ pays cash late at ni

In [36]:
# ═══════════════════════════════════════════════════════════════
# TEST 8 — TONE CONSISTENCY
# Generate 3 scenes at different points in the story
# using the accumulated bible as context
# Measure: sentence length, weather words, positive vs noir vocabulary
# Noir target profile:
#   avg sentence length 12-18 words
#   at least 1 weather reference
#   0-1 positive emotion words
#   3+ noir atmosphere words
# Variance across 3 scenes should be low (<5 words avg length diff)
# ═══════════════════════════════════════════════════════════════

current_bible = get_bible_facts(PROJECT_ID).to_dict("records")

tone_tests = [
    {
        "label": "early_tone",
        "description": "Elena searches the burned dojo for more clues. She finds a hidden trapdoor.",
        "prior_scene": TEST_SCENES[0]["content"]
    },
    {
        "label": "mid_tone",
        "description": "Tomas tells Elena that Vera has disappeared. Elena decides she needs to find the Mechanic.",
        "prior_scene": TEST_SCENES[3]["content"]
    },
    {
        "label": "late_tone",
        "description": "Elena confronts the boy about who gave him the photograph. He is more frightened than she expected.",
        "prior_scene": TEST_SCENES[7]["content"]
    },
]

def tone_signals(text: str) -> dict:
    sentences = [s.strip() for s in text.split('.') if len(s.strip()) > 5]
    avg_len = sum(len(s.split()) for s in sentences) / max(len(sentences), 1)
    weather = len(re.findall(
        r'\b(rain|fog|cold|grey|gray|dark|wet|damp|snow|cloud|smoke|ash|mist|drizzle)\b',
        text, re.I))
    positive = len(re.findall(
        r'\b(happy|smiled|laughed|warm|bright|wonderful|cheerful|joy|hope|light|glad|pleased)\b',
        text, re.I))
    noir = len(re.findall(
        r'\b(dark|cold|silence|alone|still|empty|nothing|dead|grey|shadow|quiet|hollow|numb)\b',
        text, re.I))
    return {
        "avg_sentence_length": round(avg_len, 1),
        "weather_words": weather,
        "positive_emotion": positive,
        "noir_atmosphere": noir,
        "sentence_count": len(sentences)
    }

print("── TEST 8: TONE CONSISTENCY ──\n")
tone_outputs = []

for i, test in enumerate(tone_tests):
    output = generate_scene(test["description"], current_bible, test["prior_scene"])
    signals = tone_signals(output)
    tone_outputs.append({
        "label": test["label"],
        "output": output,
        "signals": signals
    })

    len_ok = 12 <= signals["avg_sentence_length"] <= 22
    weather_ok = signals["weather_words"] >= 1
    positive_ok = signals["positive_emotion"] <= 2
    noir_ok = signals["noir_atmosphere"] >= 2

    print(f"Scene: {test['label']}")
    print(f"  {'✓' if len_ok else '✗'} Avg sentence length:  {signals['avg_sentence_length']} words  (target: 12-22)")
    print(f"  {'✓' if weather_ok else '✗'} Weather words:        {signals['weather_words']}             (target: 1+)")
    print(f"  {'✓' if positive_ok else '✗'} Positive emotion:     {signals['positive_emotion']}             (target: 0-2)")
    print(f"  {'✓' if noir_ok else '✗'} Noir atmosphere:      {signals['noir_atmosphere']}             (target: 2+)")
    print(f"\n  Text:\n  {output[:300]}...\n")

    if i < len(tone_tests) - 1:
        time.sleep(3)

avg_lengths = [t["signals"]["avg_sentence_length"] for t in tone_outputs]
variance = max(avg_lengths) - min(avg_lengths)

print(f"── CONSISTENCY CHECK ──")
print(f"Sentence length across 3 scenes: {avg_lengths}")
print(f"Variance: {variance:.1f} words  (target: <6)")
print(f"TARGET: consistent noir profile across all 3 scenes")
print(f"RESULT: {'✓ PASS' if variance < 6 else '✗ FAIL — tone drifting, add tone anchor to generation prompt'}")

── TEST 8: TONE CONSISTENCY ──

Scene: early_tone
  ✓ Avg sentence length:  19.3 words  (target: 12-22)
  ✓ Weather words:        2             (target: 1+)
  ✓ Positive emotion:     0             (target: 0-2)
  ✓ Noir atmosphere:      2             (target: 2+)

  Text:
  As Elena stepped into the smoldering dojo, the acrid smell of char and dampness enveloped her, a noxious shroud that clung to her skin and hair. The air was heavy with the scent of smoke, a constant reminder of the city's perpetual haze. She moved cautiously, her eyes scanning the ruined space for a...

Scene: mid_tone
  ✓ Avg sentence length:  17.3 words  (target: 12-22)
  ✓ Weather words:        2             (target: 1+)
  ✓ Positive emotion:     0             (target: 0-2)
  ✗ Noir atmosphere:      1             (target: 2+)

  Text:
  The dim lighting of the Cerny Kohout bar seemed to swallow Elena whole as she slid onto a stool, her eyes locking onto Tomas's familiar face. The smell of smoke and worn leather e

In [37]:
# ═══════════════════════════════════════════════════════════════
# TEST 9 — CROSS-SCENE CONTINUITY
# Fact planted in scene 1: scar on LEFT hand
# Scene 6 explicitly confirms: Elena reopened the LEFT hand wound
# Two checks:
#   A — did bible preserve "left hand" from scene 1?
#   B — does generation use "left hand" when writing a hand injury?
# This tests the full chain: extraction → bible storage → generation use
# ═══════════════════════════════════════════════════════════════

print("── TEST 9: CROSS-SCENE CONTINUITY ──\n")

df = get_bible_facts(PROJECT_ID)

# Check A — is left hand in the bible at all?
bible_strings = df["content"].apply(content_to_string)
left_hand_rows = df[bible_strings.str.contains("left", case=False)]
passed_a = len(left_hand_rows) > 0

print("Check A — was 'left hand' preserved in bible from scene 1?")
print(f"  Result: {'✓ YES' if passed_a else '✗ NO — extraction missed the scar location'}")
if passed_a:
    for _, row in left_hand_rows.iterrows():
        print(f"  Found in [{row['category']}]: {content_to_string(row['content'])[:100]}")

# Check B — does generation use left hand in a hand injury scene?
print("\nCheck B — does generation correctly use 'left hand' for injury scene?")
print("  Generating scene with hand injury description...\n")

continuity_description = """Elena cuts her hand badly on a broken window frame while 
escaping through a building. She has to bandage it herself in a stairwell. 
It is the same hand as always."""

current_bible = get_bible_facts(PROJECT_ID).to_dict("records")
continuity_output = generate_scene(
    continuity_description,
    current_bible,
    TEST_SCENES[4]["content"]
)

print("Generated scene:")
print(continuity_output)

left_mentioned = bool(re.search(r'\bleft\b', continuity_output, re.IGNORECASE))
hand_mentioned = bool(re.search(r'\bhand\b', continuity_output, re.IGNORECASE))
scar_mentioned = bool(re.search(r'\bscar\b', continuity_output, re.IGNORECASE))
passed_b = left_mentioned and hand_mentioned

print(f"\nCheck B results:")
print(f"  {'✓' if left_mentioned else '✗'} 'left' mentioned in output")
print(f"  {'✓' if hand_mentioned else '✗'} 'hand' mentioned in output")
print(f"  {'✓' if scar_mentioned else '✗'} 'scar' mentioned (bonus — shows deep bible use)")

print(f"\nCheck A (bible preserved the fact):   {'✓ PASS' if passed_a else '✗ FAIL'}")
print(f"Check B (generation used the fact):   {'✓ PASS' if passed_b else '✗ FAIL'}")
print(f"\nTARGET: both A and B pass")
print(f"OVERALL: {'✓ PASS' if passed_a and passed_b else '✗ FAIL — continuity chain has a gap'}")

if not passed_a:
    print("\nFix: strengthen extraction prompt to capture physical details explicitly")
if passed_a and not passed_b:
    print("\nFix: generation is not reading bible carefully enough — strengthen system prompt")

── TEST 9: CROSS-SCENE CONTINUITY ──

Check A — was 'left hand' preserved in bible from scene 1?
  Result: ✓ YES
  Found in [character]: elena's father was quiet and careful; habitually left rooms unnoticed.
  Found in [established_fact]: elena's father habitually left rooms unnoticed.
  Found in [character]: elena frequently injures her left hand/palm; the scar tissue on her left palm is weak and prone to r
  Found in [character]: elena protagonist (implied) red hair; has an old, pale scar across her left palm. none none

Check B — does generation correctly use 'left hand' for injury scene?
  Generating scene with hand injury description...

Generated scene:
The rain-soaked streets were a blur as Elena fled through the deserted alleys, her feet pounding against the wet pavement. She ducked into a narrow doorway, the building's entrance a dimly lit recess that seemed to swallow her whole. The air inside was musty, heavy with the scent of decay, and Elena's eyes struggled to adjust to t

In [38]:
# ═══════════════════════════════════════════════════════════════
# FULL ITERATION SUMMARY
# Fill in each result after running the test above it
# Use this to track what changed between prompt iterations
# ═══════════════════════════════════════════════════════════════

ITERATION_NUMBER = 1

print(f"╔══════════════════════════════════════════════════════════════╗")
print(f"║         ITERATION {ITERATION_NUMBER} — FULL TEST RESULTS SUMMARY           ║")
print(f"╠══════════════════════════════════════════════════════════════╣")

results = [
    ("6", "JSON Reliability",        "___/8  (target 8/8)       "),
    ("1", "Extraction Recall",        "___%   (target 80%+)      "),
    ("2", "Redundancy",               "___ pairs (target 0)      "),
    ("3", "Hallucination",            "___ invented (target 0)   "),
    ("4", "Accumulation",             "flattens? Y/N             "),
    ("5", "Bible Usage",              "___/7  (target 4+)        "),
    ("7", "Implicit Facts",           "___/4  (target 2+)        "),
    ("8", "Tone Consistency",         "variance ___w (target <6) "),
    ("9", "Cross-scene Continuity",   "A: Y/N   B: Y/N           "),
]

for num, name, result in results:
    print(f"║  Test {num}: {name:<28} {result}║")

print(f"╠══════════════════════════════════════════════════════════════╣")
print(f"║  NOTES                                                       ║")
print(f"╚══════════════════════════════════════════════════════════════╝")

print(f"""
Iteration {ITERATION_NUMBER} notes:
  Extraction prompt: [describe version]
  Generation prompt: [describe version]
  
  What failed:
  - 
  
  What to change for iteration {ITERATION_NUMBER + 1}:
  - (change ONE thing at a time)
  
  To re-run: 
    1. Edit prompt variables in the Prompts cell
    2. Re-run Prompts cell + Functions cell
    3. Run Setup cell (clears DB, re-runs all 8 scenes)
    4. Run tests 1-9 in order
    5. Update ITERATION_NUMBER above and fill in results
""")

╔══════════════════════════════════════════════════════════════╗
║         ITERATION 1 — FULL TEST RESULTS SUMMARY           ║
╠══════════════════════════════════════════════════════════════╣
║  Test 6: JSON Reliability             ___/8  (target 8/8)       ║
║  Test 1: Extraction Recall            ___%   (target 80%+)      ║
║  Test 2: Redundancy                   ___ pairs (target 0)      ║
║  Test 3: Hallucination                ___ invented (target 0)   ║
║  Test 4: Accumulation                 flattens? Y/N             ║
║  Test 5: Bible Usage                  ___/7  (target 4+)        ║
║  Test 7: Implicit Facts               ___/4  (target 2+)        ║
║  Test 8: Tone Consistency             variance ___w (target <6) ║
║  Test 9: Cross-scene Continuity       A: Y/N   B: Y/N           ║
╠══════════════════════════════════════════════════════════════╣
║  NOTES                                                       ║
╚══════════════════════════════════════════════════════════════╝

